# Araseの電磁場データについて、64 Hzデータを用いる。

# FACの定義および磁場の衛星スピントーンの補正の付加

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データの取得

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import pyspedas as psp
import pytplot as pt
import ergpyspedas.erg as ergpy

pt.del_data('*')

# Input may be either 'YYYYMMDD/HH:MM:SS' or ISO8601.
time_range_input = ['2022-09-06T01:23:26.792515938', '2022-09-06T01:55:48.782759']

def normalize_time_range(time_range_like):
    normalized = [str(t).replace('/', 'T') for t in time_range_like]
    times = pd.to_datetime(normalized, format='ISO8601')
    if times[0] >= times[1]:
        raise ValueError(f'time_range must be increasing: {time_range_like}')
    return [ts.isoformat() for ts in times], times[0], times[1]

time_range, t_start_pd, t_end_pd = normalize_time_range(time_range_input)
time_range_T = list(time_range)
t_start = np.datetime64(t_start_pd.to_datetime64())
t_end = np.datetime64(t_end_pd.to_datetime64())

folder_time_label = f"{t_start_pd:%Y-%m-%d}/{t_start_pd:%H%M}-{t_end_pd:%H%M}"
path_base_save_plot_0 = f'/mnt/j/statistical_analysis_arase/preanalysis/KAW_observation/E_B_ratio_Arase/{folder_time_label}'

def load_erg_eb_tplot(time_range):
    ergpy.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True)
    ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True)
    return psp.tplot_names()

loaded_tplot_vars = load_erg_eb_tplot(time_range)
print('--- Loaded tplot variables ---')
print(loaded_tplot_vars)


In [ ]:
def get_tplot_da(name, *, time_slice=None):
    if time_slice is None:
        time_slice = slice(time_range_T[0], time_range_T[1])
    da = psp.get_data(name, xarray=True)
    if da is None:
        raise KeyError(f'tplot variable not found: {name}')
    return da.sortby('time').sel(time=time_slice)

def _component(da, index):
    dim = 'v_dim' if 'v_dim' in da.dims else da.dims[-1]
    return da.isel({dim: index})

def load_b64_dsi(qf_max=21):
    B64_raw = get_tplot_da('erg_mgf_l2_mag_64hz_dsi')
    qf = get_tplot_da('erg_mgf_l2_quality_64hz')

    qf_B, B64 = xr.align(_component(qf, 3), B64_raw, join='inner')
    B64_qf = xr.where(qf_B <= qf_max, B64, np.nan)

    ds = xr.Dataset({
        'B64_dsi_x': _component(B64_qf, 0),
        'B64_dsi_y': _component(B64_qf, 1),
        'B64_dsi_z': _component(B64_qf, 2),
    }).dropna(dim='time', how='all')

    ds.attrs.update({
        'source': 'erg_mgf_l2_mag_64hz_dsi',
        'quality_flag': 'erg_mgf_l2_quality_64hz[:, 3]',
        'quality_rule': f'quality_flag <= {qf_max}',
    })
    return ds, B64_raw, qf, qf_B

ds_B64_dsi, B64_data_dsi, B64_data_dsi_quality_flag, qf_B = load_b64_dsi(qf_max=21)
print(ds_B64_dsi)


In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's')):
    if ds is None or time_dim not in ds.coords:
        raise ValueError(f'dataset must have a {time_dim!r} coordinate')

    ds = ds.sortby(time_dim)
    t = ds[time_dim].values
    if t.size == 0:
        return []

    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]
    seg_id = np.cumsum(gaps)

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    return [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]


In [ ]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(30, 'm'))
if not ds_B64_dsi_segs:
    raise ValueError('No valid B64 DSI samples after quality filtering')
print(f'B64 DSI segments: {len(ds_B64_dsi_segs)}')


# 磁場のスピントーン除去 [Imajo et al., 2021]

In [ ]:
da_mgf_spin_phase_deg = get_tplot_da('erg_mgf_l2_spin_phase_64hz')
print(da_mgf_spin_phase_deg)


In [ ]:
import sys, importlib
from pathlib import Path

importlib.invalidate_caches()
module_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'module_handmade').is_dir()), None)
if module_root is None:
    raise FileNotFoundError('Could not find module_handmade in cwd or its parents')
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)

SPINTONE_MIN_POINTS = int(64.0 * 3.0 / 2.0)

def remove_spintone_b64_segment(ds_B_seg, seg_index, min_points=SPINTONE_MIN_POINTS):
    phase_rad = np.deg2rad(da_mgf_spin_phase_deg.interp(time=ds_B_seg.time))
    ds_input = ds_B_seg.assign(phase_rad=phase_rad).dropna(dim='time', how='any')
    if ds_input.sizes.get('time', 0) < min_points:
        print(f'skip B64 segment {seg_index}: only {ds_input.sizes.get("time", 0)} finite samples')
        return None

    B_clean_ndarray, B_spt_ndarray, params = emsr.remove_spintone_3comp(
        time=ds_input.time.values,
        Bx=ds_input['B64_dsi_x'].values,
        By=ds_input['B64_dsi_y'].values,
        Bz=ds_input['B64_dsi_z'].values,
        phase_rad=ds_input['phase_rad'].values,
        min_points=min_points,
    )

    coords = {'time': ds_input.time.values}
    ds_spt = xr.Dataset({
        'B64_dsi_x_spt': ('time', B_spt_ndarray[:, 0]),
        'B64_dsi_y_spt': ('time', B_spt_ndarray[:, 1]),
        'B64_dsi_z_spt': ('time', B_spt_ndarray[:, 2]),
    }, coords=coords).assign_attrs(seg_index=seg_index)

    ds_clean = xr.Dataset({
        'B64_dsi_x_clean': ('time', B_clean_ndarray[:, 0]),
        'B64_dsi_y_clean': ('time', B_clean_ndarray[:, 1]),
        'B64_dsi_z_clean': ('time', B_clean_ndarray[:, 2]),
    }, coords=coords).assign_attrs(seg_index=seg_index)

    return ds_clean, ds_spt, params

ds_B64_dsi_clean_segs = []
ds_B64_dsi_spt_segs = []
spintone_params_by_seg = []

for i_seg, ds_B_seg in enumerate(ds_B64_dsi_segs):
    result = remove_spintone_b64_segment(ds_B_seg, i_seg)
    if result is None:
        continue
    ds_clean, ds_spt, params = result
    ds_B64_dsi_clean_segs.append(ds_clean)
    ds_B64_dsi_spt_segs.append(ds_spt)
    spintone_params_by_seg.append(params)

if not ds_B64_dsi_clean_segs:
    raise ValueError('No B64 DSI segment survived spin-tone removal')

ds_B64_dsi_clean = xr.concat(ds_B64_dsi_clean_segs, dim='time').sortby('time')
ds_B64_dsi_spt = xr.concat(ds_B64_dsi_spt_segs, dim='time').sortby('time')

# Compatibility names for cells that still inspect the first segment explicitly.
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
ds_B64_dsi_seg0_clean = ds_B64_dsi_clean_segs[0]
ds_B64_dsi_seg0_spt = ds_B64_dsi_spt_segs[0]
da_mgf_spin_phase_deg_interp = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi_clean.time)
da_mgf_spin_phase_rad_interp = np.deg2rad(da_mgf_spin_phase_deg_interp)

print(f'B64 spin-tone cleaned segments: {len(ds_B64_dsi_clean_segs)} / {len(ds_B64_dsi_segs)}')
print(ds_B64_dsi_clean)


# 電場データと磁場データの時間を合わせる

In [ ]:
def load_e64_dsi_xy(apply_quality_flag=False):
    E64_x = get_tplot_da('erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform')
    E64_y = get_tplot_da('erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform')
    qf = get_tplot_da('erg_pwe_efd_l2_E64Hz_dsi_quality_flag')

    bad_times = qf.time.where(qf != 0, drop=True)
    if apply_quality_flag:
        qf_x = qf.interp(time=E64_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'})
        qf_y = qf.interp(time=E64_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'})
        E64_x = E64_x.where(qf_x == 0, np.nan)
        E64_y = E64_y.where(qf_y == 0, np.nan)

    ds = xr.Dataset({
        'E64_dsi_x': E64_x,
        'E64_dsi_y': E64_y,
    }).dropna(dim='time', how='all')

    ds.attrs.update({
        'source': 'erg_pwe_efd_l2_E64Hz_dsi_*_waveform',
        'quality_flag': 'erg_pwe_efd_l2_E64Hz_dsi_quality_flag',
        'quality_rule': 'quality_flag == 0' if apply_quality_flag else 'not applied',
    })
    return ds, E64_x, E64_y, qf, bad_times

APPLY_EFD_QUALITY_FLAG = False
(
    ds_E64_dsi_xy,
    E64_data_dsi_x,
    E64_data_dsi_y,
    E64_data_dsi_quality_flag,
    bad_times,
) = load_e64_dsi_xy(apply_quality_flag=APPLY_EFD_QUALITY_FLAG)

print(ds_E64_dsi_xy)
print(f'EFD bad quality samples: {bad_times.size}')


In [ ]:
ds_E64_dsi_xy_segs = split_by_gap(ds_E64_dsi_xy, gap_thr=np.timedelta64(63, 'ms'))
print(ds_E64_dsi_xy_segs)

In [ ]:
def make_ds_EB_func(ds_E_xy, E_vars, ds_B_xyz, B_vars, output_vars):
    time_base   = ds_E_xy.time
    ds_B_xyz_interp  = ds_B_xyz.interp(time=time_base, method='linear')

    da_Ex   = ds_E_xy[E_vars[0]]
    da_Ey   = ds_E_xy[E_vars[1]]
    da_Bx   = ds_B_xyz_interp[B_vars[0]]
    da_By   = ds_B_xyz_interp[B_vars[1]]
    da_Bz   = ds_B_xyz_interp[B_vars[2]]

    da_Ez   = xr.where(np.abs(da_Bz) > 1E-2, -(da_Ex * da_Bx + da_Ey * da_By) / da_Bz, np.nan)

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
E64_xy_vars     = ['E64_dsi_x', 'E64_dsi_y']
B64_xyz_vars    = ['B64_dsi_x_clean', 'B64_dsi_y_clean', 'B64_dsi_z_clean']
EB64_vars       = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

In [ ]:
def _overlap_time_range(ds_a, ds_b):
    start = max(ds_a.time.values[0], ds_b.time.values[0])
    stop = min(ds_a.time.values[-1], ds_b.time.values[-1])
    if start > stop:
        return None
    return start, stop

def make_ds_EB64_dsi_segments(ds_E_segs, ds_B_clean_segs):
    ds_EB_segs = []
    for i_E, ds_E_seg in enumerate(ds_E_segs):
        n_before = len(ds_EB_segs)
        for i_B, ds_B_seg in enumerate(ds_B_clean_segs):
            overlap = _overlap_time_range(ds_E_seg, ds_B_seg)
            if overlap is None:
                continue
            t0_overlap, t1_overlap = overlap
            ds_E_part = ds_E_seg.sel(time=slice(t0_overlap, t1_overlap))
            if ds_E_part.sizes.get('time', 0) == 0:
                continue
            ds_EB = make_ds_EB_func(ds_E_part, E64_xy_vars, ds_B_seg, B64_xyz_vars, EB64_vars)
            if ds_EB.sizes.get('time', 0) == 0:
                continue
            ds_EB = ds_EB.assign_attrs(E_seg_index=i_E, B_seg_index=i_B)
            ds_EB_segs.append(ds_EB)
        if len(ds_EB_segs) == n_before:
            print(f'skip E64 segment {i_E}: no overlapping cleaned B64 segment')
    return ds_EB_segs

ds_EB64_dsi_segs = make_ds_EB64_dsi_segments(ds_E64_dsi_xy_segs, ds_B64_dsi_clean_segs)
if not ds_EB64_dsi_segs:
    raise ValueError('No overlapping E64/B64 DSI segments after spin-tone removal')

print(f'EB64 DSI segments: {len(ds_EB64_dsi_segs)}')
print(ds_EB64_dsi_segs)


In [ ]:
path_base_save_plot = path_base_save_plot_0
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
step_min    = 1
step    = np.timedelta64(step_min, 'm')

t_list = []
t_win = t_start
while t_win < t_end:
    t_next = t_win + step
    t_list.append((t_win, t_next))
    t_win = t_next

def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['E64_dsi_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['E64_dsi_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['E64_dsi_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['B64_dsi_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['B64_dsi_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['B64_dsi_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (DSI)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (DSI)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (DSI)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (DSI)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (DSI)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (DSI)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsi'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EB64_dsi_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)


# FAC座標系を定義、DSI座標系 -> FAC座標系変換行列の作成

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。 ($E_{x}$: Toroidal component, $B_{x}$: Poloidal component)
- y軸は、z軸とx軸の外積で与える。 ($E_{y}$: Poloidal component, $B_{y}$: Toroidal component)

In [ ]:
def load_arase_position_dsi(time_range):
    ergpy.orb(trange=time_range, level='l2', datatype='def')
    pos_gsm = get_tplot_da('erg_orb_l2_pos_gsm')
    pos_unit_gsm = pos_gsm / np.sqrt((pos_gsm * pos_gsm).sum(dim='v_dim'))

    psp.store_data(
        'Arase_pos_unit_gsm',
        data={'x': pos_unit_gsm.time.values, 'y': pos_unit_gsm.data},
    )
    psp.cotrans(
        name_in='Arase_pos_unit_gsm',
        name_out='Arase_pos_unit_j2000',
        coord_in='gsm',
        coord_out='j2000',
    )
    psp.projects.erg.erg_cotrans(
        in_name='Arase_pos_unit_j2000',
        out_name='Arase_pos_unit_dsi',
        in_coord='j2000',
        out_coord='dsi',
    )
    pos_unit_dsi = psp.get_data('Arase_pos_unit_dsi', xarray=True).sortby('time')
    return pos_gsm, pos_unit_gsm, pos_unit_dsi

da_Arase_pos_gsm, da_Arase_pos_unit_gsm, da_Arase_pos_unit_dsi = load_arase_position_dsi(time_range)
print(da_Arase_pos_unit_dsi)


In [ ]:
background_time_sec = 100  # sec

In [ ]:
def rolling_background_b64_segments(ds_B_clean_segs, window_sec):
    ds_bg_segs = []
    for i_seg, ds_seg in enumerate(ds_B_clean_segs):
        n_time = ds_seg.sizes.get('time', 0)
        if n_time < 2:
            print(f'skip B64 background segment {i_seg}: fewer than 2 samples')
            continue

        dt = np.nanmedian(np.diff(ds_seg.time.values) / np.timedelta64(1, 's'))
        if not np.isfinite(dt) or dt <= 0:
            print(f'skip B64 background segment {i_seg}: invalid cadence {dt}')
            continue

        n_window = max(1, int(round(window_sec / dt)))
        if n_time < n_window:
            print(f'skip B64 background segment {i_seg}: shorter than rolling window')
            continue

        ds_bg = ds_seg.rolling(time=n_window, center=True).mean('time')
        ds_bg_segs.append(ds_bg.assign_attrs(ds_seg.attrs))

    if not ds_bg_segs:
        raise ValueError('No B64 segment is long enough for background-field rolling mean')
    return xr.concat(ds_bg_segs, dim='time').sortby('time')

def dataset_to_vector_da(ds, vector_dim='v_dim'):
    return ds.to_dataarray(dim=vector_dim).transpose('time', vector_dim)

ds_B_background = rolling_background_b64_segments(ds_B64_dsi_clean_segs, background_time_sec)
da_B_background = dataset_to_vector_da(ds_B_background).dropna(dim='time', how='any')
da_B_background_unit = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit = da_B_background_unit.dropna(dim='time', how='all')

B_background_norm = np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))
print(da_B_background_unit)
print(float(np.nanmin(B_background_norm)), float(np.nanmax(B_background_norm)))


In [ ]:
def make_fac_basis_from_b_and_position(B_unit_dsi, pos_unit_dsi, min_perp_norm=1e-6):
    pos_interp = pos_unit_dsi.interp(time=B_unit_dsi.time)
    z_fac_in_dsi = B_unit_dsi.drop_attrs()

    pos_perp = pos_interp - (pos_interp * z_fac_in_dsi).sum(dim='v_dim') * z_fac_in_dsi
    pos_perp_norm = np.sqrt((pos_perp * pos_perp).sum(dim='v_dim'))
    x_fac_in_dsi = (pos_perp / pos_perp_norm).where(pos_perp_norm > min_perp_norm).drop_attrs()
    y_fac_in_dsi = xr.apply_ufunc(
        np.cross,
        z_fac_in_dsi,
        x_fac_in_dsi,
        input_core_dims=[['v_dim'], ['v_dim']],
        output_core_dims=[['v_dim']],
        vectorize=True,
    ).drop_attrs()

    x_fac_in_dsi.name = 'x_FAC_in_DSI'
    y_fac_in_dsi.name = 'y_FAC_in_DSI'
    z_fac_in_dsi.name = 'z_FAC_in_DSI'
    return x_fac_in_dsi, y_fac_in_dsi, z_fac_in_dsi

da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI = make_fac_basis_from_b_and_position(
    da_B_background_unit,
    da_Arase_pos_unit_dsi,
)

print(da_e_x_FAC_inDSI)
print(da_e_y_FAC_inDSI)
print(da_e_z_FAC_inDSI)


In [ ]:
def make_fac_rotation_matrices(x_fac_in_dsi, y_fac_in_dsi, z_fac_in_dsi):
    R_fac_to_dsi = xr.concat(
        [x_fac_in_dsi, y_fac_in_dsi, z_fac_in_dsi],
        dim='axis',
    )
    R_fac_to_dsi = R_fac_to_dsi.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC'])
    R_fac_to_dsi = R_fac_to_dsi.assign_coords(v_dim=np.arange(3))

    R_dsi_to_fac = R_fac_to_dsi.transpose('time', 'v_dim', 'axis')
    return R_fac_to_dsi, R_dsi_to_fac

def summarize_fac_basis_quality(x_fac, y_fac, z_fac):
    pairs = {
        'x_dot_y': (x_fac * y_fac).sum(dim='v_dim'),
        'y_dot_z': (y_fac * z_fac).sum(dim='v_dim'),
        'z_dot_x': (z_fac * x_fac).sum(dim='v_dim'),
    }
    norms = {
        'x_norm': np.sqrt((x_fac * x_fac).sum(dim='v_dim')),
        'y_norm': np.sqrt((y_fac * y_fac).sum(dim='v_dim')),
        'z_norm': np.sqrt((z_fac * z_fac).sum(dim='v_dim')),
    }
    for name, da in {**pairs, **norms}.items():
        print(name, float(np.nanmedian(da)), float(np.nanmax(np.abs(da))))

R_FAC_to_DSI, R_DSI_to_FAC = make_fac_rotation_matrices(
    da_e_x_FAC_inDSI,
    da_e_y_FAC_inDSI,
    da_e_z_FAC_inDSI,
)

summarize_fac_basis_quality(da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI)
print(R_FAC_to_DSI)
print(R_DSI_to_FAC)


In [ ]:
da_e_x_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=0)
da_e_y_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=1)
da_e_z_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=2)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['font.size'] = 15

path_base_save_plot = f'{path_base_save_plot_0}/coordinate'
os.makedirs(path_base_save_plot, exist_ok=True)

PLOT_FAC_DSI_FRAME = True

def cart_to_polar(u, v):
    return np.arctan2(v, u), np.hypot(u, v)

def setup_polar_ax(ax, title):
    ax.set_title(title)
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.minorticks_on()
    ax.set_rmax(1.0)
    ax.set_rticks([0.25, 0.5, 0.75, 1.0])
    ax.set_rlabel_position(135)
    ax.grid(True)

def plot_polar_vector(ax, u, v, color, label):
    theta, r = cart_to_polar(u, v)
    ax.plot([theta, theta], [0, r], color=color, lw=2, label=label)
    ax.plot(theta, r, 'o', color=color, ms=5)

def plot_fac_dsi_frame_polar(it, frame_idx, save_dir):
    ex = da_e_x_DSI_inFAC.isel(time=it)
    ey = da_e_y_DSI_inFAC.isel(time=it)
    ez = da_e_z_DSI_inFAC.isel(time=it)

    ex_x, ex_y, ex_z = [ex.sel(axis=a).item() for a in ['x_FAC', 'y_FAC', 'z_FAC']]
    ey_x, ey_y, ey_z = [ey.sel(axis=a).item() for a in ['x_FAC', 'y_FAC', 'z_FAC']]
    ez_x, ez_y, ez_z = [ez.sel(axis=a).item() for a in ['x_FAC', 'y_FAC', 'z_FAC']]

    fig, axs = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={'projection': 'polar'})

    setup_polar_ax(axs[0], '(x, y) plane')
    plot_polar_vector(axs[0], 1, 0, 'k', 'FAC-x')
    plot_polar_vector(axs[0], 0, 1, 'gray', 'FAC-y')
    plot_polar_vector(axs[0], ex_x, ex_y, 'r', 'DSI-x')
    plot_polar_vector(axs[0], ey_x, ey_y, 'b', 'DSI-y')
    plot_polar_vector(axs[0], ez_x, ez_y, 'g', 'DSI-z')
    axs[0].legend(loc='lower left', bbox_to_anchor=(-0.15, -0.15))

    setup_polar_ax(axs[1], '(x, z) plane')
    plot_polar_vector(axs[1], 1, 0, 'k', 'FAC-x')
    plot_polar_vector(axs[1], 0, 1, 'purple', 'FAC-z')
    plot_polar_vector(axs[1], ex_x, ex_z, 'r', 'DSI-x')
    plot_polar_vector(axs[1], ey_x, ey_z, 'b', 'DSI-y')
    plot_polar_vector(axs[1], ez_x, ez_z, 'g', 'DSI-z')

    setup_polar_ax(axs[2], '(y, z) plane')
    plot_polar_vector(axs[2], 1, 0, 'gray', 'FAC-y')
    plot_polar_vector(axs[2], 0, 1, 'purple', 'FAC-z')
    plot_polar_vector(axs[2], ex_y, ex_z, 'r', 'DSI-x')
    plot_polar_vector(axs[2], ey_y, ey_z, 'b', 'DSI-y')
    plot_polar_vector(axs[2], ez_y, ez_z, 'g', 'DSI-z')

    fig.suptitle(str(da_e_x_DSI_inFAC.time.values[it]))
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f'coord_polar_{frame_idx:06d}.png'), dpi=150)
    plt.close(fig)


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from tqdm import tqdm

if PLOT_FAC_DSI_FRAME:
    n_time = da_e_x_DSI_inFAC.sizes['time']
    step = 64 * 60
    tasks = [(it, frame_idx) for frame_idx, it in enumerate(range(0, n_time, step))]
    print('num frames:', len(tasks))

    def worker(args):
        it, frame_idx, save_path = args
        plot_fac_dsi_frame_polar(it, frame_idx, save_path)
        return frame_idx

    n_workers = max(1, mp.cpu_count() - 1)
    with ProcessPoolExecutor(max_workers=n_workers) as exe:
        futures = [exe.submit(worker, (it, idx, path_base_save_plot)) for it, idx in tasks]
        for f in tqdm(as_completed(futures), total=len(futures)):
            _ = f.result()


# DSI座標系 -> FAC座標系変換の実行

In [ ]:
def rotate_eb64_dsi_to_fac(ds_EB_dsi_segs, R_dsi_to_fac):
    ds_fac_segs = []
    for i_seg, ds_seg in enumerate(ds_EB_dsi_segs):
        R_seg = R_dsi_to_fac.interp(time=ds_seg.time)

        E_dsi = xr.concat(
            [ds_seg['E64_dsi_x'], ds_seg['E64_dsi_y'], ds_seg['E64_dsi_z']],
            dim='v_dim',
        ).assign_coords(v_dim=np.arange(3))
        B_dsi = xr.concat(
            [ds_seg['B64_dsi_x'], ds_seg['B64_dsi_y'], ds_seg['B64_dsi_z']],
            dim='v_dim',
        ).assign_coords(v_dim=np.arange(3))

        E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')
        B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

        E_fac_ds = E_fac.to_dataset(dim='axis').rename({
            'x_FAC': 'E64_fac_x',
            'y_FAC': 'E64_fac_y',
            'z_FAC': 'E64_fac_z',
        })
        B_fac_ds = B_fac.to_dataset(dim='axis').rename({
            'x_FAC': 'B64_fac_x',
            'y_FAC': 'B64_fac_y',
            'z_FAC': 'B64_fac_z',
        })

        ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
        if ds_fac.sizes.get('time', 0) == 0:
            print(f'skip EB64 FAC segment {i_seg}: no finite rotated samples')
            continue
        ds_fac_segs.append(ds_fac.assign_attrs(ds_seg.attrs, dsi_seg_index=i_seg))

    if not ds_fac_segs:
        raise ValueError('No EB64 segment survived DSI to FAC rotation')
    return ds_fac_segs

ds_EB64_fac_segs = rotate_eb64_dsi_to_fac(ds_EB64_dsi_segs, R_DSI_to_FAC)
print(f'EB64 FAC segments: {len(ds_EB64_fac_segs)}')
print(ds_EB64_fac_segs[0])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

path_base_save_plot = path_base_save_plot_0
os.makedirs(path_base_save_plot, exist_ok=True)

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
step_min    = 1
step    = np.timedelta64(step_min, 'm')

t_list = []
t_win = t_start
while t_win < t_end:
    t_next = t_win + step
    t_list.append((t_win, t_next))
    t_win = t_next

def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['E64_fac_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['E64_fac_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['E64_fac_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['B64_fac_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['B64_fac_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['B64_fac_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EB64_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)


# 軌道データから、衛星速度(DSI)を導出

In [ ]:
def derive_spacecraft_velocity_gse():
    pos_gse = get_tplot_da('erg_orb_l2_pos_gse')  # R_E
    t_pos = pos_gse.time.values

    R_E_m = 6.378137e6
    t_s = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
    v_gse = np.gradient(pos_gse.data * R_E_m, t_s, axis=0)

    v_sc = xr.DataArray(
        data=v_gse,
        dims=('time', 'v_dim'),
        coords={'time': t_pos, 'v_dim': np.arange(3)},
        attrs={'units': 'm/s', 'desc': '$V_{sc}$ (GSE)'},
        name='v_sc_gse',
    )
    return v_sc

v_sc_gse = derive_spacecraft_velocity_gse()
print(v_sc_gse)


In [ ]:
path_base_save_plot = path_base_save_plot_0

In [ ]:
import matplotlib.pyplot as plt
import datetime

time_range_analysis     = time_range
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

v_sc_gse_analysis  = v_sc_gse.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(3, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 0]*1E-3, lw=1, c='k')
ax_1.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 1]*1E-3, lw=1, c='k')
ax_2.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 2]*1E-3, lw=1, c='k')

ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSE)' + '\n' + '[km/s]')
ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSE)' + '\n' + '[km/s]')
ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSE)' + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)

ax_2.set_xlim(v_sc_gse_analysis.time.values[[0, -1]])

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'v_sc_gse.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

def store_vector_tplot(name, da):
    psp.store_data(
        name,
        data={
            'x': da.time.values.astype('datetime64[ns]'),
            'y': np.asarray(da.data, dtype=np.float64),
        },
    )

def transform_v_sc_to_dsi(v_sc_gse):
    store_vector_tplot('v_sc_gse_64', v_sc_gse)
    psp.cotrans(name_in='v_sc_gse_64', name_out='v_sc_j2000_64', coord_in='gse', coord_out='j2000')
    dsi2j2000(name_in='v_sc_j2000_64', name_out='v_sc_dsi_64', J20002DSI=True)

    v_sc_j2000 = psp.get_data('v_sc_j2000_64', xarray=True).sortby('time')
    v_sc_dsi = psp.get_data('v_sc_dsi_64', xarray=True).sortby('time')
    return v_sc_j2000, v_sc_dsi

def check_vector_norm_preserved(a, b, tag, thr_rel=1e-5):
    a_aligned, b_aligned = xr.align(a, b, join='inner')
    na = np.linalg.norm(a_aligned.data, axis=1)
    nb = np.linalg.norm(b_aligned.data, axis=1)
    finite = np.isfinite(na) & np.isfinite(nb)
    if not finite.any():
        raise ValueError(f'{tag}: no finite samples after coordinate transform')
    rel = np.abs(na[finite] - nb[finite]) / np.maximum(na[finite], 1e-30)
    print(f'{tag}: finite={finite.sum()}/{finite.size}, rel mean={np.nanmean(rel):.3e}, max={np.nanmax(rel):.3e}')
    if np.nanmax(rel) >= thr_rel:
        raise ValueError(f'{tag} norm not preserved above tolerance {thr_rel:g}')

v_sc_j2000, v_sc_dsi = transform_v_sc_to_dsi(v_sc_gse)
check_vector_norm_preserved(v_sc_gse, v_sc_j2000, 'GSE -> J2000')
check_vector_norm_preserved(v_sc_j2000, v_sc_dsi, 'J2000 -> DSI')


In [ ]:
import matplotlib.pyplot as plt
import datetime

time_range_analysis     = time_range
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

v_sc_j2000_analysis  = v_sc_j2000.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(3, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 0]*1E-3, lw=1, c='k')
ax_1.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 1]*1E-3, lw=1, c='k')
ax_2.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 2]*1E-3, lw=1, c='k')

ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (J2000)' + '\n' + '[km/s]')
ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (J2000)' + '\n' + '[km/s]')
ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (J2000)' + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)

ax_2.set_xlim(v_sc_j2000_analysis.time.values[[0, -1]])

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'v_sc_j2000.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import datetime

time_range_analysis     = time_range
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

v_sc_dsi_analysis  = v_sc_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(3, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
ax_1.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
ax_2.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')

ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (DSI)' + '\n' + '[km/s]')
ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (DSI)' + '\n' + '[km/s]')
ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (DSI)' + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)

ax_2.set_xlim(v_sc_dsi_analysis.time.values[[0, -1]])

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'v_sc_dsi.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
def rotate_vector_dsi_to_fac(da_dsi, R_dsi_to_fac, name=None):
    R_interp = R_dsi_to_fac.interp(time=da_dsi.time)
    da_fac = xr.dot(da_dsi.assign_coords(v_dim=np.arange(3)), R_interp, dims='v_dim')
    da_fac = da_fac.dropna(dim='time', how='any')
    if name is not None:
        da_fac.name = name
    return da_fac

v_sc_fac = rotate_vector_dsi_to_fac(v_sc_dsi, R_DSI_to_FAC, name='v_sc_fac')
print(v_sc_fac)


In [ ]:
import matplotlib.pyplot as plt
import datetime

time_range_analysis     = time_range
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(4, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')

ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)

ax_3.set_xlim(v_sc_fac_analysis.time.values[[0, -1]])

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
def store_background_magnetic_field_for_lep(da_B_background, name='erg_mgf_l2_mag_64hz_background_dsi'):
    psp.store_data(
        name,
        data={'x': da_B_background.time.values, 'y': da_B_background.data},
    )
    return name

background_mag_tplot_name = store_background_magnetic_field_for_lep(da_B_background)
print(background_mag_tplot_name)


In [ ]:
import pyspedas as psp
import pytplot as pt

ergpy.lepe(trange=time_range, datatype='3dflux', level='l2')
ergpy.lepi(trange=time_range, datatype='3dflux', level='l2')
ergpy.pwe_hfa(trange=time_range, level='l3')

# LEP-i moments are partial moments over this measured E/q range.
# H+, He+, and O+ are singly charged, so the energy argument in eV is
# numerically equal to E/q in eV/q.
LEPI_PARTIAL_ENERGY_RANGE_EVQ = [30.0, 3.0e4]

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name=background_mag_tplot_name,
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    energy=LEPI_PARTIAL_ENERGY_RANGE_EVQ,
    mag_name=background_mag_tplot_name,
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FHEDU',
    outputs=['moments'],
    energy=LEPI_PARTIAL_ENERGY_RANGE_EVQ,
    mag_name=background_mag_tplot_name,
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FODU',
    outputs=['moments'],
    energy=LEPI_PARTIAL_ENERGY_RANGE_EVQ,
    mag_name=background_mag_tplot_name,
    pos_name='erg_orb_l2_pos_gse'
)


In [ ]:

proton_mass_kg = 1.6726219e-27  # kg
Helium_mass_kg = 4.0 * proton_mass_kg
Oxygen_mass_kg = 16.0 * proton_mass_kg
electron_mass_kg = 9.1093837e-31  # kg
elementary_charge = 1.60217663e-19  # C
mu0 = 4.0 * np.pi * 1e-7
eps0 = 8.8541878188e-12

ION_SPECIES = {
    'proton': {
        'density': 'erg_lepi_l2_3dflux_FPDU_density',
        'flux': 'erg_lepi_l2_3dflux_FPDU_flux',
        'temp': 'erg_lepi_l2_3dflux_FPDU_avgtemp',
        'mass': proton_mass_kg,
    },
    'Helium': {
        'density': 'erg_lepi_l2_3dflux_FHEDU_density',
        'flux': 'erg_lepi_l2_3dflux_FHEDU_flux',
        'temp': 'erg_lepi_l2_3dflux_FHEDU_avgtemp',
        'mass': Helium_mass_kg,
    },
    'Oxygen': {
        'density': 'erg_lepi_l2_3dflux_FODU_density',
        'flux': 'erg_lepi_l2_3dflux_FODU_flux',
        'temp': 'erg_lepi_l2_3dflux_FODU_avgtemp',
        'mass': Oxygen_mass_kg,
    },
}

def get_tplot_sorted(name, *, fillna=None):
    da = psp.get_data(name, xarray=True)
    if da is None:
        raise KeyError(f'tplot variable not found: {name}')
    da = da.sortby('time')
    return da.fillna(fillna) if fillna is not None else da

def load_electron_moments():
    ne_lep = get_tplot_sorted('erg_lepe_l2_3dflux_FEDU_density')
    te = get_tplot_sorted('erg_lepe_l2_3dflux_FEDU_avgtemp')

    ne_hfa = get_tplot_sorted('erg_pwe_hfa_l3_1min_ne_mgf')
    ne_hfa_flag = get_tplot_sorted('erg_pwe_hfa_l3_1min_quality_flag')
    ne_hfa = xr.where(ne_hfa_flag < 1, ne_hfa, np.nan).sortby('time')
    return ne_lep, ne_hfa, ne_hfa_flag, te

def load_ion_species_moments(species_config):
    out = {}
    for species, cfg in species_config.items():
        out[species] = {
            # NaN denotes an unavailable moment, not a physical zero.
            'density': get_tplot_sorted(cfg['density']),
            'flux': get_tplot_sorted(cfg['flux']),
            'temp': get_tplot_sorted(cfg['temp']),
            'mass': cfg['mass'],
        }
    return out

def combine_ion_moments(ion_moments):
    species = list(ion_moments)
    species_coord = xr.IndexVariable('species', species)
    densities = xr.concat(
        [ion_moments[s]['density'] for s in species],
        dim=species_coord, join='outer',
    )
    fluxes = xr.concat(
        [ion_moments[s]['flux'] for s in species],
        dim=species_coord, join='outer',
    )
    temperatures = xr.concat(
        [ion_moments[s]['temp'] for s in species],
        dim=species_coord, join='outer',
    )
    masses = xr.DataArray(
        [ion_moments[s]['mass'] for s in species],
        dims='species', coords={'species': species},
    )

    # Density and mean mass use species with a finite, positive density.
    valid_density = np.isfinite(densities) & (densities > 0)
    densities_valid = densities.where(valid_density)
    n_species_density_used = valid_density.sum(dim='species')
    nd_ion = densities_valid.sum(dim='species', skipna=True, min_count=1)
    ion_mass = (
        (densities_valid * masses).sum(dim='species', skipna=True, min_count=1)
        / nd_ion
    )

    # Temperature uses the same valid species in pressure and density sums.
    valid_temperature = (
        valid_density & np.isfinite(temperatures) & (temperatures >= 0)
    )
    n_species_temperature_used = valid_temperature.sum(dim='species')
    p_tot_ion = (densities * temperatures).where(valid_temperature).sum(
        dim='species', skipna=True, min_count=1,
    )
    nd_for_temperature = densities.where(valid_temperature).sum(
        dim='species', skipna=True, min_count=1,
    )
    temp_ion = p_tot_ion / nd_for_temperature

    # A species contributes to bulk velocity only when its density and every
    # vector-flux component are finite. Numerator and denominator then use
    # exactly the same species subset.
    flux_component_dims = [
        dim for dim in fluxes.dims if dim not in {'species', 'time'}
    ]
    if not flux_component_dims:
        raise ValueError(f'Ion flux has no vector-component dimension: {fluxes.dims}')
    valid_flux = valid_density & np.isfinite(fluxes).all(dim=flux_component_dims)
    n_species_flux_used = valid_flux.sum(dim='species')
    flux_ion = fluxes.where(valid_flux).sum(
        dim='species', skipna=True, min_count=1,
    )
    nd_for_velocity = densities.where(valid_flux).sum(
        dim='species', skipna=True, min_count=1,
    )
    v_ion_dsi = flux_ion / nd_for_velocity * 1e-2  # cm/s -> m/s

    partial_note = (
        'Partial LEP-i moment over 30 eV/q <= E/q <= 30 keV/q; '
        'missing species are excluded, not replaced by zero.'
    )
    for da in [nd_ion, flux_ion, p_tot_ion, v_ion_dsi, temp_ion, ion_mass]:
        da.attrs['partial_moment_note'] = partial_note
        da.attrs['energy_range_eV_per_q'] = LEPI_PARTIAL_ENERGY_RANGE_EVQ

    return (
        nd_ion, flux_ion, p_tot_ion, v_ion_dsi, temp_ion, ion_mass,
        n_species_density_used, n_species_temperature_used, n_species_flux_used,
    )

ND_electron_LEP, ND_electron_HFA, ND_electron_HFA_flag, Temp_electron = load_electron_moments()
ion_moments = load_ion_species_moments(ION_SPECIES)

ND_proton = ion_moments['proton']['density']
Flux_proton = ion_moments['proton']['flux']
Temp_proton = ion_moments['proton']['temp']
ND_Helium = ion_moments['Helium']['density']
Flux_Helium = ion_moments['Helium']['flux']
Temp_Helium = ion_moments['Helium']['temp']
ND_Oxygen = ion_moments['Oxygen']['density']
Flux_Oxygen = ion_moments['Oxygen']['flux']
Temp_Oxygen = ion_moments['Oxygen']['temp']

(
    ND_ion, Flux_ion, Ptot_ion, v_ion_dsi, Temp_ion, ion_mass,
    n_species_density_used, n_species_temperature_used, n_species_flux_used,
) = combine_ion_moments(ion_moments)

print(ND_ion)
print(v_ion_dsi)


In [ ]:

v_ion_fac = rotate_vector_dsi_to_fac(v_ion_dsi, R_DSI_to_FAC, name='v_ion_fac')
print(v_ion_fac.time)
print(v_sc_fac.time)


In [ ]:

def make_system_velocity_fac(v_ion_fac, v_sc_fac):
    v_sys = v_ion_fac.interp(time=v_sc_fac.time, method='linear') - v_sc_fac
    v_sys = v_sys.dropna(dim='time', how='any')
    v_sys.name = 'v_sys_fac'
    return v_sys

v_sys_fac = make_system_velocity_fac(v_ion_fac, v_sc_fac)
print(v_sys_fac)


In [ ]:
PLOT_PRE_WAVELET_DIAGNOSTICS = True

if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    v_ion_dsi_analysis  = v_ion_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(3, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
    ax_1.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
    ax_2.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')

    ax_0.set_ylabel(r'$V_{\mathrm{i}x}$ (DSI)' + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$V_{\mathrm{i}y}$ (DSI)' + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$V_{\mathrm{i}z}$ (DSI)' + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)

    ax_2.set_xlim(v_ion_dsi_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'v_ion_dsi.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(4, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
    ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
    ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
    ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')

    ax_0.set_ylabel(r'$V_{\mathrm{i}x}$ (FAC)' + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$V_{\mathrm{i}y}$ (FAC)' + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$V_{\mathrm{i}z}$ (FAC)' + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$V_{\mathrm{i}\perp}$ (FAC)' + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)

    ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(4, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
    ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
    ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
    ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')

    ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)

    ax_3.set_xlim(v_sys_fac_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
    v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
    v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(4, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0]*1E-3, lw=1, c='k')
    ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1]*1E-3, lw=1, c='k')
    ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2]*1E-3, lw=1, c='k')
    ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0)*1E-3, lw=1, c='k')

    ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)

    ax_3.set_xlim(v_sys_fac_analysis_mean.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    ND_proton_analysis  = ND_proton.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_Helium_analysis  = ND_Helium.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_Oxygen_analysis  = ND_Oxygen.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_ion_analysis     = ND_ion.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ion_mass_analysis  = ion_mass.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(5, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(ND_proton_analysis.time,  ND_proton_analysis.data,    lw=1, c='k')
    ax_1.plot(ND_Helium_analysis.time,  ND_Helium_analysis.data,    lw=1, c='k')
    ax_2.plot(ND_Oxygen_analysis.time,  ND_Oxygen_analysis.data,    lw=1, c='k')
    ax_3.plot(ND_ion_analysis.time,     ND_ion_analysis.data,       lw=1, c='k')
    ax_4.plot(ion_mass_analysis.time,   ion_mass_analysis.data/proton_mass_kg, lw=1, c='k')

    ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_0.set_yscale('log')
    #ax_0.set_ylim(ymax=2)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_1.set_yscale('log')
    #ax_1.set_ylim(ymin=1E-4)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_2.set_yscale('log')
    #ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_3.set_yscale('log')
    #ax_3.set_ylim(ymax=2)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)
    #ax_4.set_ylim(ymin=1, ymax=2.5)

    ax_4.set_xlim(ion_mass_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    dt_ND               = (ND_proton.time.data[1] - ND_proton.time.data[0]) / np.timedelta64(1, 's')

    ND_proton_mean  = ND_proton.rolling(time=int(100/dt_ND), center=True).mean()
    ND_Helium_mean  = ND_Helium.rolling(time=int(100/dt_ND), center=True).mean()
    ND_Oxygen_mean  = ND_Oxygen.rolling(time=int(100/dt_ND), center=True).mean()
    ND_ion_mean     = ND_ion.rolling(time=int(100/dt_ND), center=True).mean()
    ion_mass_mean   = ion_mass.rolling(time=int(100/dt_ND), center=True).mean()

    ND_proton_analysis_mean  = ND_proton_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_Helium_analysis_mean  = ND_Helium_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_Oxygen_analysis_mean  = ND_Oxygen_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ND_ion_analysis_mean     = ND_ion_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ion_mass_analysis_mean   = ion_mass_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(5, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(ND_proton_analysis_mean.time,  ND_proton_analysis_mean.data,    lw=1, c='k')
    ax_1.plot(ND_Helium_analysis_mean.time,  ND_Helium_analysis_mean.data,    lw=1, c='k')
    ax_2.plot(ND_Oxygen_analysis_mean.time,  ND_Oxygen_analysis_mean.data,    lw=1, c='k')
    ax_3.plot(ND_ion_analysis_mean.time,     ND_ion_analysis_mean.data,       lw=1, c='k')
    ax_4.plot(ion_mass_analysis_mean.time,   ion_mass_analysis_mean.data/proton_mass_kg, lw=1, c='k')

    ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_0.set_yscale('log')
    #ax_0.set_ylim(ymax=2)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_1.set_yscale('log')
    #ax_1.set_ylim(ymin=1E-4)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_2.set_yscale('log')
    #ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_3.set_yscale('log')
    #ax_3.set_ylim(ymax=2)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)
    #ax_4.set_ylim(ymin=1, ymax=2.5)

    ax_4.set_xlim(ion_mass_analysis_mean.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND_mean.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


# HFAチェック

In [ ]:
ergpy.pwe_hfa(trange=time_range, level='l2')
da_hfa_spectral = psp.get_data('erg_pwe_hfa_l2_low_spectra_esum', xarray=True)

In [ ]:

def make_hfa_characteristic_frequencies(B_total_nT, n_e_cc):
    n_e_interp = n_e_cc.interp(time=B_total_nT.time, method='linear')
    f_ce = elementary_charge * B_total_nT * 1e-9 / electron_mass_kg / (2.0 * np.pi) / 1e3
    f_pe = np.sqrt(n_e_interp * 1e6 * elementary_charge**2 / electron_mass_kg / eps0) / (2.0 * np.pi) / 1e3
    f_uhr = np.sqrt(f_ce**2 + f_pe**2)
    f_ce.name = 'f_ce'
    f_pe.name = 'f_pe'
    f_uhr.name = 'f_UHR'
    return f_uhr, f_ce, f_pe

B_total_for_hfa = np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_f_UHR, da_f_ce, da_f_pe = make_hfa_characteristic_frequencies(B_total_for_hfa, ND_electron_HFA)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    from datetime import datetime
    import matplotlib.dates as mdates
    from matplotlib.colors import LogNorm
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    da_hfa_spectral_analysis    = da_hfa_spectral.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_f_UHR_analysis           = da_f_UHR.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_f_ce_analysis            = da_f_ce.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_f_pe_analysis            = da_f_pe.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    fig = plt.figure(figsize=(12, 5))

    ax_0    = fig.add_subplot(111)

    T_0 = mdates.date2num(da_hfa_spectral_analysis.time.values)
    F_0 = da_hfa_spectral_analysis.spec_bins.values
    Z_0 = da_hfa_spectral_analysis.values.astype(float)

    T_0_m = np.tile(T_0, (F_0.size, 1)).T
    F_0_m = np.tile(F_0, (T_0.size, 1))

    if len(T_0_m) != 0:
        pcm = ax_0.pcolormesh(T_0_m, F_0_m, Z_0, shading='auto', norm=LogNorm(vmin=1E-8, vmax=1E-4), cmap='turbo')
        ax_0.plot(da_f_UHR_analysis.time, da_f_UHR_analysis.data, c='white', lw=2)
        ax_0.minorticks_on()
        ax_0.set_yscale('log')
        ax_0.set_ylim(5E0, 5E1)
        ax_0.set_ylabel('PWE-HFA [kHz]')
        ax_0.grid(which='both', alpha=0.5)

        time_range_T_analysis_0, time_range_T_analysis_1 = t_start, t_end
        ax_0.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

        cax = ax_0.inset_axes([1.01, 0.05, 0.02, 0.9])
        cb = plt.colorbar(pcm, cax=cax)
        cb.minorticks_on()
        cb.set_label(r'$[\mathrm{(mV/m)}^{2} / \mathrm{Hz}]$')

        fig.tight_layout()

        if os.path.isdir(path_base_save_plot):
            fig_path = os.path.join(path_base_save_plot, 'PWE-HFA_spec.png')
            print(fig_path)
            fig.savefig(fig_path)
            plt.close(fig)
        else:
            plt.show()
            plt.close(fig)
    else:
        plt.close(fig)


- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [ ]:

da_B64_dsi_clean = ds_B64_dsi_clean.to_dataarray(dim='v_dim').assign_coords(v_dim=np.arange(3))
da_B64_dsi_seg0_clean = ds_B64_dsi_seg0_clean.to_dataarray(dim='v_dim').assign_coords(v_dim=np.arange(3))

da_B64_dsi_clean_total = np.sqrt((da_B64_dsi_clean * da_B64_dsi_clean).sum(dim='v_dim'))
da_B64_dsi_seg0_clean_total = np.sqrt((da_B64_dsi_seg0_clean * da_B64_dsi_seg0_clean).sum(dim='v_dim'))

print(da_B64_dsi_clean_total)


In [ ]:

def dedup_and_sort(obj):
    obj = obj.sortby('time')
    t = obj['time'].values
    _, keep = np.unique(t, return_index=True)
    return obj.isel(time=np.sort(keep))

def interp_unique_time(da, time_base, method='linear'):
    return dedup_and_sort(da).interp(time=time_base, method=method)

def rolling_mean_by_seconds(da, window_sec, time_dim='time'):
    n_time = da.sizes.get(time_dim, 0)
    if n_time < 2:
        return da * np.nan
    dt = np.nanmedian(np.diff(da[time_dim].values) / np.timedelta64(1, 's'))
    if not np.isfinite(dt) or dt <= 0:
        return da * np.nan
    n_window = max(1, int(round(window_sec / dt)))
    return da.rolling({time_dim: n_window}, center=True).mean()

def vector_component(da_vec, index, label=None):
    if 'v_dim' in da_vec.dims:
        return da_vec.isel(v_dim=index)
    if 'axis' in da_vec.dims:
        axis_values = set(da_vec.coords['axis'].values.tolist()) if 'axis' in da_vec.coords else set()
        if label is not None and label in axis_values:
            return da_vec.sel(axis=label)
        return da_vec.isel(axis=index)
    vector_dims = [dim for dim in da_vec.dims if dim != 'time']
    if not vector_dims:
        raise ValueError(f'No vector component dimension found in dims={da_vec.dims}')
    return da_vec.isel({vector_dims[0]: index})

def vector_perp_magnitude(da_vec):
    x = vector_component(da_vec, index=0, label='x_FAC')
    y = vector_component(da_vec, index=1, label='y_FAC')
    out = np.sqrt(x**2 + y**2)
    out.attrs.update(da_vec.attrs)
    return out


In [ ]:

def build_plasma_parameter_products(time_base, smoothing_sec=100.0):
    B_total = interp_unique_time(da_B64_dsi_clean_total, time_base)

    ne_lep = interp_unique_time(ND_electron_LEP, time_base)
    ne_hfa = interp_unique_time(ND_electron_HFA, time_base)
    ne_mid = (ne_hfa + ne_lep) / 2.0
    te = interp_unique_time(Temp_electron, time_base)

    ti = interp_unique_time(Temp_ion, time_base)
    mi = interp_unique_time(ion_mass, time_base)
    vi_fac = interp_unique_time(v_ion_fac, time_base)
    vsys_fac = interp_unique_time(v_sys_fac, time_base)

    vi_perp = vector_perp_magnitude(vi_fac)
    vsys_perp = vector_perp_magnitude(vsys_fac)
    vi_x = vector_component(vi_fac, index=0, label='x_FAC')
    vi_y = vector_component(vi_fac, index=1, label='y_FAC')
    vsys_x = vector_component(vsys_fac, index=0, label='x_FAC')
    vsys_y = vector_component(vsys_fac, index=1, label='y_FAC')

    vA_lep = B_total * 1e-9 / np.sqrt(mu0 * ne_lep * 1e6 * mi)
    vA_hfa = B_total * 1e-9 / np.sqrt(mu0 * ne_hfa * 1e6 * mi)
    vA_mid = B_total * 1e-9 / np.sqrt(mu0 * ne_mid * 1e6 * mi)

    vthi = np.sqrt(2.0 * ti * elementary_charge / mi)
    cs = np.sqrt(te * elementary_charge / mi)
    vthe = np.sqrt(2.0 * te * elementary_charge / electron_mass_kg)
    fcp = elementary_charge * B_total * 1e-9 / proton_mass_kg / (2.0 * np.pi)

    beta_lep = (vthi / vA_lep)**2
    beta_hfa = (vthi / vA_hfa)**2
    beta_mid = (vthi / vA_mid)**2
    ti_te_ratio = (vthi / cs)**2 / 2.0

    def sm(da):
        return rolling_mean_by_seconds(da, smoothing_sec)

    common_velocity = {
        'Alfven_speed_LEP': sm(vA_lep),
        'Alfven_speed_MID': sm(vA_mid),
        'Alfven_speed_HFA': sm(vA_hfa),
        'ion_thermal_speed': sm(vthi),
        'electron_thermal_speed': sm(vthe),
        'ion_acoustic_speed': sm(cs),
    }

    ds_velocity_ms_perp = xr.Dataset({
        **common_velocity,
        'perp_ion_speed': sm(vi_perp),
        'perp_sys_speed': sm(vsys_perp),
    }).dropna(dim='time', how='all')

    ds_velocity_ms_toroidal = xr.Dataset({
        **common_velocity,
        'perp_ion_speed': sm(vi_x),
        'perp_sys_speed': sm(vsys_x),
    }).dropna(dim='time', how='all')

    ds_velocity_ms_poloidal = xr.Dataset({
        **common_velocity,
        'perp_ion_speed': sm(vi_y),
        'perp_sys_speed': sm(vsys_y),
    }).dropna(dim='time', how='all')

    ds_parameter = xr.Dataset({
        'ion_plasma_beta_LEP': sm(beta_lep),
        'ion_plasma_beta_MID': sm(beta_mid),
        'ion_plasma_beta_HFA': sm(beta_hfa),
        'i-e_temp_ratio': sm(ti_te_ratio),
        'proton_cycl_freq_Hz': sm(fcp),
        'number_density_LEP_cc': sm(ne_lep),
        'number_density_MID_cc': sm(ne_mid),
        'number_density_HFA_cc': sm(ne_hfa),
        'ion_mass_kg': sm(mi),
        'temp_ion_eV': sm(ti),
        'temp_electron_eV': sm(te),
        'B_total_nT': sm(B_total),
    }).dropna(dim='time', how='all')

    return ds_parameter, ds_velocity_ms_perp, ds_velocity_ms_toroidal, ds_velocity_ms_poloidal

time_base = da_B64_dsi_clean_total.time
(
    ds_parameter,
    ds_velocity_ms_perp,
    ds_velocity_ms_toroidal,
    ds_velocity_ms_poloidal,
) = build_plasma_parameter_products(time_base, smoothing_sec=100.0)

print(ds_parameter)
print(ds_velocity_ms_perp)


In [ ]:
PLOT_GEOMAG_INDICES = True
GEOMAG_NEAREST_TOLERANCE = None  # e.g. np.timedelta64(90, 'm'); None keeps nearest values over the loaded range.


def parse_time_like(t):
    if isinstance(t, np.datetime64):
        return pd.Timestamp(t)
    text = str(t)
    if '/' in text:
        return pd.to_datetime(text, format='%Y%m%d/%H:%M:%S')
    return pd.Timestamp(np.datetime64(text))


def format_pyspedas_time(t):
    return parse_time_like(t).strftime('%Y%m%d/%H:%M:%S')


def expand_time_range_for_geomag(t0, t1, margin_hours=3):
    t0_np = np.datetime64(t0) - np.timedelta64(margin_hours, 'h')
    t1_np = np.datetime64(t1) + np.timedelta64(margin_hours, 'h')
    return [format_pyspedas_time(t0_np), format_pyspedas_time(t1_np)]


def get_first_tplot_da(candidates):
    names = set(psp.tplot_names())
    for name in candidates:
        if name in names:
            da = psp.get_data(name, xarray=True)
            if da is not None:
                da = da.sortby('time')
                if da.ndim > 1:
                    da = da.squeeze(drop=True)
                return da
    return None


def omni_hourly_date_trange(trange):
    # pyspedas OMNI hourly uses yearlynames(), whose parser expects YYYY-MM-DD.
    # The end date is exclusive after time_clip, so add one day to keep hourly samples after midnight.
    start = parse_time_like(trange[0]).floor('D').strftime('%Y-%m-%d')
    end = (parse_time_like(trange[1]).floor('D') + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    return [start, end]


def load_geomagnetic_indices(trange):
    # 1-min OMNI contains AE/AU/AL/AO and SYM-H-like high-cadence indices when available.
    psp.projects.omni.data(trange=trange, datatype='1min')
    # Hourly OMNI is the usual source for Dst-like hourly index. It requires date-only trange.
    # Use a prefix to avoid overwriting 1-min AL_INDEX/AU_INDEX tplot variables.
    psp.projects.omni.data(trange=omni_hourly_date_trange(trange), datatype='hourly', prefix='omni_hourly_')

    candidates = {
        'AE': ['AE_INDEX', 'AE', 'ae_index', 'ae'],
        'AU': ['AU_INDEX', 'AU', 'au_index', 'au'],
        'AL': ['AL_INDEX', 'AL', 'al_index', 'al'],
        'AO': ['AO_INDEX', 'AO', 'ao_index', 'ao'],
        'SYM_H': ['SYM_H', 'SYM-H', 'sym_h', 'SYM_H_INDEX'],
        'Dst': ['omni_hourly_DST', 'omni_hourly_DST_INDEX', 'omni_hourly_Dst', 'DST_INDEX', 'DST', 'Dst', 'dst', 'DST_NT'],
    }

    raw = {}
    for label, names in candidates.items():
        da = get_first_tplot_da(names)
        if da is None:
            print(f'skip geomagnetic index {label}: no matching tplot variable in {names}')
            continue
        # Keep the margin-loaded native cadence here; hourly Dst often has no sample inside a short event window.
        da = da.sel(time=slice(parse_time_like(trange[0]).to_datetime64(), parse_time_like(trange[1]).to_datetime64()))
        da.name = f'geomag_{label}_nT'
        da.attrs.update({'units': 'nT', 'source': 'OMNI via pyspedas', 'native_index': label})
        raw[label] = da
    if not raw:
        raise ValueError('No geomagnetic indices were loaded from OMNI')
    return xr.Dataset({da.name: da for da in raw.values()})


def attach_geomagnetic_indices(ds_parameter, ds_geomag_raw, tolerance=None):
    ds_out = ds_parameter.copy()
    for name, da in ds_geomag_raw.data_vars.items():
        da_valid = da.dropna(dim='time', how='all')
        if da_valid.sizes.get('time', 0) == 0:
            print(f'skip {name}: all values are NaN after native-cadence cleanup')
            continue
        da_sorted = dedup_and_sort(da_valid)
        reindex_kwargs = {'method': 'nearest'}
        if tolerance is not None:
            reindex_kwargs['tolerance'] = tolerance
        aligned = da_sorted.reindex(time=ds_parameter.time, **reindex_kwargs)
        aligned.attrs.update(da.attrs)
        aligned.attrs['time_alignment'] = 'nearest to ds_parameter.time'
        ds_out[name] = aligned
    return ds_out


geomag_trange = expand_time_range_for_geomag(t_start, t_end, margin_hours=3)
ds_geomag_indices_raw = load_geomagnetic_indices(geomag_trange)
ds_parameter = attach_geomagnetic_indices(ds_parameter, ds_geomag_indices_raw, tolerance=GEOMAG_NEAREST_TOLERANCE)

print(ds_geomag_indices_raw)
print(ds_parameter[[name for name in ds_parameter.data_vars if name.startswith('geomag_')]])


In [ ]:
def plot_geomagnetic_indices(ds_geomag_raw, ds_parameter_aligned=None, save_dir=None):
    if ds_geomag_raw is None or not ds_geomag_raw.data_vars:
        print('No geomagnetic index dataset to plot')
        return

    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import matplotlib.dates as mdates

    def plot_index_da(name):
        da = None
        if name in ds_geomag_raw:
            da_raw = ds_geomag_raw[name].dropna(dim='time', how='all').sel(
                time=slice(np.datetime64(t_start), np.datetime64(t_end))
            )
            if da_raw.sizes.get('time', 0) > 0:
                da = da_raw
        if da is None and ds_parameter_aligned is not None and name in ds_parameter_aligned:
            da = ds_parameter_aligned[name].dropna(dim='time', how='all').sel(
                time=slice(np.datetime64(t_start), np.datetime64(t_end))
            )
            n_time = da.sizes.get('time', 0)
            if n_time > 2000:
                da = da.isel(time=slice(None, None, int(np.ceil(n_time / 2000))))
        return da

    mpl.rcParams['font.size'] = 14
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, height_ratios=[2, 1])

    ae_colors = {
        'geomag_AL_nT': 'tab:red',
        'geomag_AU_nT': 'tab:blue',
        'geomag_AE_nT': 'purple',
        'geomag_AO_nT': 'tab:green',
    }
    for name, color in ae_colors.items():
        da = plot_index_da(name)
        if da is not None and da.sizes.get('time', 0) > 0:
            axes[0].plot(da.time, da, lw=1.2, color=color, label=name.replace('geomag_', '').replace('_nT', ''))
    axes[0].set_ylabel('AE indices\n[nT]')
    axes[0].legend(ncol=4, fontsize=11)
    axes[0].minorticks_on()
    axes[0].grid(which='both', alpha=0.35)

    for name, color, label in [
        ('geomag_Dst_nT', 'k', 'Dst'),
        ('geomag_SYM_H_nT', 'tab:orange', 'SYM-H'),
    ]:
        da = plot_index_da(name)
        if da is not None and da.sizes.get('time', 0) > 0:
            axes[1].plot(da.time, da, lw=1.4, color=color, label=label)
        else:
            print(f'skip plot {label}: no finite values in plot range')
    axes[1].axhline(0, color='0.5', lw=0.8, ls='--')
    axes[1].set_ylabel('Dst / SYM-H\n[nT]')
    axes[1].set_xlabel('Time')
    axes[1].legend(ncol=2, fontsize=11)
    axes[1].minorticks_on()
    axes[1].grid(which='both', alpha=0.35)
    axes[1].set_xlim(np.datetime64(t_start), np.datetime64(t_end))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    fig.tight_layout()
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        for suffix in ('png', 'pdf'):
            fig_path = save_dir / f'Arase_geomagnetic_indices.{suffix}'
            print(fig_path)
            fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


if PLOT_GEOMAG_INDICES:
    plot_geomagnetic_indices(ds_geomag_indices_raw, ds_parameter_aligned=ds_parameter, save_dir=Path(path_base_save_plot_0))


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import matplotlib.ticker as mticker
    from datetime import datetime
    import matplotlib.dates as mdates

    mpl.rcParams['font.size'] = 25

    fig = plt.figure(figsize=(11, 21))
    gs = fig.add_gridspec(7, 1, hspace=0)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
    ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)
    ax_5.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_LEP_cc'],       lw=2, c='b', linestyle='-.', label=r'LEP-e')
    ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],       lw=1, c='k', label=r'MID')
    ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_HFA_cc'],       lw=2, c='r', linestyle='-.', label=r'HFA')
    ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
    ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
    ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k')
    ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],         lw=1, c='k', label=r'$\beta_{\mathrmi}}$')
    ax_3.plot(ds_parameter_analysis.time, electron_mass_kg / ds_parameter_analysis['ion_mass_kg'], lw=2, c='green', linestyle='-.', label=r'$m_{\mathrm{e}}/m_{\mathrm{i}}$')
    ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='b', label=r'$v_{\mathrmthe}}$')
    ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,   lw=1, c='k', label=r'$v_{\mathrm{A}}$')
    ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='red')

    ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + r'[$\mathrm{cm}^{-3}$]')
    ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
    ax_2.set_ylabel(r'$B_{0}$'                              + '\n' + '[nT]')
    ax_3.set_ylabel(r'$\beta_{\mathrm{i}}$')
    ax_4.set_ylabel(r'$v_{\mathrm{the}}$'                   + '\n' + '[km/s]')
    ax_5.set_ylabel(r'$v_{\mathrm{A}}$'                     + '\n' + '[km/s]')
    ax_6.set_ylabel(r'$v_{\mathrm{thi}}$'                   + '\n' + '[km/s]')

    ax_0.set_yscale('log')
    ax_1.set_yscale('log')
    ax_3.set_yscale('log')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)
    ax_5.minorticks_on()
    ax_5.grid(which='both', alpha=0.5)
    ax_6.minorticks_on()
    ax_6.grid(which='both', alpha=0.5)

    ax_0.legend(fontsize=20, ncol=3)
    ax_1.legend(fontsize=25, ncol=2)

    time_range_T_analysis_0, time_range_T_analysis_1 = t_start, t_end
    ax_6.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
    ax_6.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    if time_range_T_analysis_1 - time_range_T_analysis_0 <= np.timedelta64(1, 'h'):
        ax_6.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))
    else:
        ax_6.xaxis.set_major_locator(mdates.MinuteLocator(interval=20))

    def add_panel_label(ax, label, x=-0.15, y=0.95):
        ax.text(x, y, label, transform=ax.transAxes,
                ha='right', va='bottom', clip_on=False)

    def to_py_datetime(t_np64):
        return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

    # 軌道データ
    pos_da = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)  # (Nt, 3)
    t_pos_py = to_py_datetime(pos_da.time.values)
    t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

    pos_X   = np.asarray(pos_da[:, 0], dtype=float)
    pos_Y   = np.asarray(pos_da[:, 1], dtype=float)
    pos_Z   = np.asarray(pos_da[:, 2], dtype=float)

    # 補間関数（tick の x は「日数」なのでそのまま使う）
    def interp_at(x_num):
        pos_X_i = np.interp(x_num, t_pos_num, pos_X, left=np.nan, right=np.nan)
        pos_Y_i = np.interp(x_num, t_pos_num, pos_Y, left=np.nan, right=np.nan)
        pos_Z_i = np.interp(x_num, t_pos_num, pos_Z, left=np.nan, right=np.nan)
        return pos_X_i, pos_Y_i, pos_Z_i

    # 目盛フォーマッタ
    def pos_formatter(x, pos=None):
        pos_X_i, pos_Y_i, pos_Z_i = interp_at(x)
        if np.any(~np.isfinite([pos_X_i, pos_Y_i, pos_Z_i])):
            return ""  # 範囲外は空
        return (f"{pos_X_i:0.2f}\n"
                f"{pos_Y_i:0.2f}\n"
                f"{pos_Z_i:0.2f}")

    # セカンダリ x 軸（底 side）を作ってラベルを差し替え
    secax = ax_6.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
    secax.xaxis.set_major_formatter(mticker.FuncFormatter(pos_formatter))

    # メインの時間ラベルと重ならないよう余白を広げる
    ax_6.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
    secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

    # 好みで：目盛間隔をメイン x と合わせる
    secax.set_ticks(ax_6.get_xticks())

    fig.text(0.07, 0.067, "hhmm", ha='center', va='center')
    fig.text(0.07, 0.047, r"X-GSM", ha='center', va='center')
    fig.text(0.07, 0.027, r"Y-GSM", ha='center', va='center')
    fig.text(0.07, 0.007, r"Z-GSM", ha='center', va='center')

    ax_0.set_title('Arase')

    fig.subplots_adjust(hspace=0)
    fig.tight_layout(pad=0)

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'Arase_parameter.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        fig.savefig(os.path.join(path_base_save_plot, 'Arase_parameter.pdf'), bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    import matplotlib.pyplot as plt
    import matplotlib as mpl

    mpl.rcParams['font.size'] = 15

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(5, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
    ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
    ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
    ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
    ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')

    ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
    ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_0.set_yscale('log')
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)

    ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'velocity_summary_perp.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    ds_velocity_ms_analysis = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    import matplotlib.pyplot as plt
    import matplotlib as mpl

    mpl.rcParams['font.size'] = 15

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(5, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
    ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
    ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
    ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
    ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')

    ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
    ax_4.set_ylabel(r'$V_{\mathrm{sys}x}$'  + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_0.set_yscale('log')
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)

    ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'velocity_summary_toroidal.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    ds_velocity_ms_analysis = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    import matplotlib.pyplot as plt
    import matplotlib as mpl

    mpl.rcParams['font.size'] = 15

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(5, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)

    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       s=1E-4, c='b')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       s=1E-4, c='k')
    ax_0.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       s=1E-4, c='r')
    ax_1.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      s=1E-4, c='k')
    ax_2.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, s=1E-4, c='k')
    ax_3.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     s=1E-4, c='k')
    ax_4.scatter(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         s=1E-4, c='k')

    ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
    ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
    ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
    ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
    ax_4.set_ylabel(r'$V_{\mathrm{sys}y}$'  + '\n' + '[km/s]')

    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.5)
    ax_0.set_yscale('log')
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.5)
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.5)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.5)
    ax_4.minorticks_on()
    ax_4.grid(which='both', alpha=0.5)

    ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'velocity_summary_poloidal.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:

ds_parameter_clean = dedup_and_sort(ds_parameter)
ds_velocity_ms_perp_clean = dedup_and_sort(ds_velocity_ms_perp)
ds_velocity_ms_poloidal_clean = dedup_and_sort(ds_velocity_ms_poloidal)
ds_velocity_ms_toroidal_clean = dedup_and_sort(ds_velocity_ms_toroidal)

ds_parameter_interp = ds_parameter_clean.interp(time=ds_velocity_ms_perp_clean.time)
print(ds_parameter_interp)
print(ds_velocity_ms_perp_clean)


# KAWが期待される周波数の最小値$f_{\mathrm{predict}}$の検討

```math
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
f_{\mathrm{predict}} := 1 \times f_{\mathrm{ci}} \frac{V_{\mathrm{sys}\perp}}{v_{\mathrm{thi}}}
```

In [ ]:

def calc_f_predict(ds_parameter, ds_velocity):
    return (
        ds_parameter['proton_cycl_freq_Hz']
        * ds_velocity['perp_sys_speed']
        / ds_velocity['ion_thermal_speed']
    )

da_f_predict = calc_f_predict(ds_parameter_clean, ds_velocity_ms_perp_clean)
da_f_predict.name = 'f_predict'


In [ ]:
time_range_analysis     = time_range
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_f_predict_window))

In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    mpl.rcParams['font.size'] = 15
    fig = plt.figure(figsize=(10, 4))
    ax = fig.add_subplot(111)
    ax.plot(da_f_predict_window.time, da_f_predict_window.data, c='k', lw=1)
    ax.minorticks_on()
    ax.set_ylabel(r'$f_{\mathrm{predict}}$' + '\n[Hz]')
    ax.grid(which='both', alpha=0.5)
    ax.set_xlim(t_start, t_end)

    ax.set_yscale('log')

    fig.tight_layout()


    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'f_predict.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


# Taylor's hypothesisが成り立つ平行波長 $\Lambda_{\parallel}$ [Hows et al., 2014]
```math
\Lambda_{\parallel} = \frac{2\pi}{\left| k_{\parallel} \right|} \gg \frac{v_{\mathrm{thi}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{A}}}{\left| V_{\mathrm{sys}\perp} \right|} \sqrt{\frac{1}{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}
```

In [ ]:

def calc_lambda_parallel(ds_parameter, ds_velocity):
    f_ci = ds_parameter['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter['ion_mass_kg']
    return (
        ds_velocity['Alfven_speed_MID']
        * ds_velocity['ion_thermal_speed']
        / np.abs(ds_velocity['perp_sys_speed'])
        / f_ci
        * np.sqrt(0.5 * (1.0 + 1.0 / ds_parameter['i-e_temp_ratio']))
    )

da_Lambda_para_toroidal = calc_lambda_parallel(ds_parameter_clean, ds_velocity_ms_toroidal_clean)
da_Lambda_para_poloidal = calc_lambda_parallel(ds_parameter_clean, ds_velocity_ms_poloidal_clean)
da_Lambda_para_toroidal.name = 'Lambda_para_toroidal'
da_Lambda_para_poloidal.name = 'Lambda_para_poloidal'


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from datetime import datetime

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    da_Lambda_para_toroidal_analysis    = da_Lambda_para_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_perp_sys_speed_toroidal_analysis = ds_velocity_ms_toroidal_clean['perp_sys_speed'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    print(da_perp_sys_speed_toroidal_analysis)

    mpl.rcParams['font.size'] = 15
    fig = plt.figure(figsize=(10, 6))

    ax_0 = fig.add_subplot(211)
    ax_0.plot(da_perp_sys_speed_toroidal_analysis.time, da_perp_sys_speed_toroidal_analysis.data * 1E-3, c='k', lw=1)
    ax_0.axhline(0, c='grey', linestyle='dashed', alpha=0.7)
    ax_0.minorticks_on()
    ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ [km/s]')
    ax_0.grid(which='both', alpha=0.5)
    ax_0.tick_params(axis='x', which='both', labelbottom=False)

    ax_1 = fig.add_subplot(212, sharex=ax_0)
    ax_1.plot(da_Lambda_para_toroidal_analysis.time, da_Lambda_para_toroidal_analysis.data / (6378.1*1E3), c='k', lw=1)
    ax_1.minorticks_on()
    ax_1.set_ylabel(r'$\Lambda_{\parallel}^{E_{x} B_{y}}$ [$R_{\mathrm{E}}$]')
    ax_1.grid(which='both', alpha=0.5)

    time_range_T_analysis_0, time_range_T_analysis_1 = t_start, t_end
    ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
    ax_1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    #ax_1.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

    ax_1.set_yscale('log')
    ax_1.set_ylim(ymin=1, ymax=100)

    add_panel_label(ax_0, '(c)', x=-0.05)
    add_panel_label(ax_1, '(d)', x=-0.05)

    ax_0.set_title('Arase')

    fig.tight_layout()

    print(np.nanmin(da_Lambda_para_toroidal_analysis), np.nanmean(da_Lambda_para_toroidal_analysis))

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'Arase_min_Lambda_para_toroidal.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        fig.savefig(os.path.join(path_base_save_plot, 'Arase_min_Lambda_para_toroidal.pdf'), bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


```math
\left| k_{x} \right| \rho_{\mathrm{i}} = 2 \pi \\
\left| k_{x} V_{\mathrm{sys}x} \right| = \left( 2 \pi \right)^{2} f_{\mathrm{ci}} \frac{\left| V_{\mathrm{sys}x} \right|}{v_{\mathrm{thi}}}
```

In [ ]:

def calc_kperp_vsys(ds_parameter, ds_velocity):
    f_ci = ds_parameter['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter['ion_mass_kg']
    return (2.0 * np.pi)**2 * f_ci * np.abs(ds_velocity['perp_sys_speed']) / ds_velocity['ion_thermal_speed']

da_kperp_Vsys_toroidal = calc_kperp_vsys(ds_parameter_clean, ds_velocity_ms_toroidal_clean)
da_kperp_Vsys_poloidal = calc_kperp_vsys(ds_parameter_clean, ds_velocity_ms_poloidal_clean)
da_ion_cycl_freq = ds_parameter_clean['proton_cycl_freq_Hz'] * proton_mass_kg / ds_parameter_clean['ion_mass_kg']

da_kperp_Vsys_toroidal.name = 'kperp_Vsys_toroidal'
da_kperp_Vsys_poloidal.name = 'kperp_Vsys_poloidal'
da_ion_cycl_freq.name = 'ion_cycl_freq'


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from datetime import datetime

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    da_kperp_Vsys_toroidal_analysis     = da_kperp_Vsys_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_perp_sys_speed_toroidal_analysis = ds_velocity_ms_toroidal_clean['perp_sys_speed'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_ion_cycl_freq_analysis   = da_ion_cycl_freq.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    print(da_perp_sys_speed_toroidal_analysis)

    mpl.rcParams['font.size'] = 15
    fig = plt.figure(figsize=(10, 6))

    ax_0 = fig.add_subplot(211)
    ax_0.plot(da_perp_sys_speed_toroidal_analysis.time, da_perp_sys_speed_toroidal_analysis.data * 1E-3, c='k', lw=1)
    ax_0.axhline(0, c='grey', linestyle='dashed', alpha=0.7)
    ax_0.minorticks_on()
    ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ [km/s]')
    ax_0.grid(which='both', alpha=0.5)
    ax_0.tick_params(axis='x', which='both', labelbottom=False)

    ax_1 = fig.add_subplot(212, sharex=ax_0)
    ax_1.plot(da_kperp_Vsys_toroidal_analysis.time, da_kperp_Vsys_toroidal_analysis.data / (2 * np.pi), c='k', lw=1)
    ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data, c='red', lw=1)
    ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data / 10., c='red', lw=2, linestyle='dashed')
    ax_1.minorticks_on()
    ax_1.set_ylabel(r'$\left| k_{x} V_{\mathrm{sys}x} \right| / 2 \pi$ [Hz]' + '\n' + r'($\left| k_{x} \right| \rho_{\mathrm{i}} = 2 \pi$)')
    ax_1.grid(which='both', alpha=0.5)

    time_range_T_analysis_0, time_range_T_analysis_1 = t_start, t_end
    ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
    ax_1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    #ax_1.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

    ax_1.set_yscale('log')
    ax_1.set_ylim(ymin=0.01, ymax=64)

    add_panel_label(ax_0, '(c)', x=-0.05)
    add_panel_label(ax_1, '(d)', x=-0.05)

    ax_0.set_title('Arase')

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_toroidal.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        fig.savefig(os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_toroidal.pdf'), bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


In [ ]:
if globals().get('PLOT_PRE_WAVELET_DIAGNOSTICS', False):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from datetime import datetime

    time_range_analysis     = time_range
    time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

    da_kperp_Vsys_poloidal_analysis     = da_kperp_Vsys_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_perp_sys_speed_poloidal_analysis = ds_velocity_ms_poloidal_clean['perp_sys_speed'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
    da_ion_cycl_freq_analysis   = da_ion_cycl_freq.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

    print(da_perp_sys_speed_poloidal_analysis)

    mpl.rcParams['font.size'] = 15
    fig = plt.figure(figsize=(10, 6))

    ax_0 = fig.add_subplot(211)
    ax_0.plot(da_perp_sys_speed_poloidal_analysis.time, da_perp_sys_speed_poloidal_analysis.data * 1E-3, c='k', lw=1)
    ax_0.axhline(0, c='grey', linestyle='dashed', alpha=0.7)
    ax_0.minorticks_on()
    ax_0.set_ylabel(r'$V_{\mathrm{sys}y}$ [km/s]')
    ax_0.grid(which='both', alpha=0.5)
    ax_0.tick_params(axis='x', which='both', labelbottom=False)

    ax_1 = fig.add_subplot(212, sharex=ax_0)
    ax_1.plot(da_kperp_Vsys_poloidal_analysis.time, da_kperp_Vsys_poloidal_analysis.data / (2 * np.pi), c='k', lw=1)
    ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data, c='red', lw=1)
    ax_1.plot(da_ion_cycl_freq_analysis.time, da_ion_cycl_freq_analysis.data / 10., c='red', lw=2, linestyle='dashed')
    ax_1.minorticks_on()
    ax_1.set_ylabel(r'$\left| k_{x} V_{\mathrm{sys}y} \right| / 2 \pi$ [Hz]' + '\n' + r'($\left| k_{y} \right| \rho_{\mathrm{i}} = 2 \pi$)')
    ax_1.grid(which='both', alpha=0.5)

    time_range_T_analysis_0, time_range_T_analysis_1 = t_start, t_end
    ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
    ax_1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    #ax_1.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

    ax_1.set_yscale('log')
    ax_1.set_ylim(ymin=0.01, ymax=64)

    #add_panel_label(ax_0, '(c)', x=-0.05)
    #add_panel_label(ax_1, '(d)', x=-0.05)

    ax_0.set_title('Arase')

    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        fig_path = os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_poloidal.png')
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        fig.savefig(os.path.join(path_base_save_plot, 'Arase_kperp_Vsys_poloidal.pdf'), bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)


# Wavelet analysis

In [ ]:
import concurrent.futures as cf
import gc
import importlib
import os
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pywt
import xarray as xr
from matplotlib.colors import LogNorm, Normalize, SymLogNorm

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

mpl.rcParams['font.size'] = 11

WAVELET_FS_HZ = 64.0
WAVELET_DT_SEC = 1.0 / WAVELET_FS_HZ
WAVELET_S0 = 2.0
WAVELET_DJ = 1.0 / 32.0
WAVELET_TARGET_F_MIN_HZ = 0.01
WAVELET_MIN_CYCLES = 3.0
WAVELET_OVERWRITE = False
WAVELET_N_JOBS = 8  # Increase cautiously; CWT/XWT arrays are memory-heavy.
PLOT_WAVELET_DIAGNOSTICS = True
PLOT_WAVELET_CWT_PANELS = True
PLOT_WAVELET_XWT_PANELS = True
PLOT_WAVELET_SPARA_PANELS = True
PLOT_WAVELET_MEDIAN_PSD = True
WAVELET_PLOT_MAX_TIME_POINTS = 2500
WAVELET_PLOT_GC_EVERY = 1

wavelet_save_dir = Path(
    f"/mnt/j/observation_data/statistical_analysis_arase/"
    f"Arase_analysis_save_data/wavelet_spectra/{folder_time_label}"
)
wavelet_cache_dir = wavelet_save_dir / "mc_cache"
wavelet_plot_dir = Path(f"/mnt/j/statistical_analysis_arase/preanalysis/KAW_observation/E_B_ratio_Arase/{folder_time_label}/wavelet_PSD")
wavelet_save_dir.mkdir(parents=True, exist_ok=True)
wavelet_cache_dir.mkdir(parents=True, exist_ok=True)
wavelet_plot_dir.mkdir(parents=True, exist_ok=True)
path_base_save_plot = str(wavelet_plot_dir)


def wavelet_segment_path(i_seg):
    return wavelet_save_dir / f"Arase_cwt_xwt_fs{WAVELET_FS_HZ:.4f}_seg{i_seg}.nc"


def existing_wavelet_segment_paths(ds_segs):
    paths = [wavelet_segment_path(i_seg) for i_seg in range(len(ds_segs))]
    return paths, all(path.exists() for path in paths)


def make_time_windows(t0, t1, step=np.timedelta64(5, 'm')):
    return list(np.arange(np.datetime64(t0), np.datetime64(t1), step))

time_windows_5min = make_time_windows(t_start, t_end)


In [ ]:
def summarize_segment_durations(ds_segs):
    durations = []
    for i, ds_seg in enumerate(ds_segs):
        n_points = ds_seg.sizes.get('time', 0)
        if n_points < 2:
            print(f'seg_{i}: n_points={n_points}, skipped for duration estimate')
            continue
        duration_sec = (ds_seg.time.values[-1] - ds_seg.time.values[0]) / np.timedelta64(1, 's')
        durations.append(float(duration_sec))
        print(f'seg_{i}: n_points={n_points}, duration={duration_sec:.3f} s')
    if not durations:
        raise ValueError('No non-empty EB64 FAC segments for wavelet analysis')
    return durations


def choose_wavelet_j_max(durations_sec, *, fs, dt, s0, dj, target_f_min, min_cycles):
    t_max = max(durations_sec)
    f_min_phys = min_cycles / t_max
    f_min = max(target_f_min, f_min_phys)
    scale_target = 1.0 / (f_min * dt)
    j_max = int(np.ceil(np.log2(scale_target / s0) / dj))
    return t_max, f_min, j_max


wavelet_segment_durations_sec = summarize_segment_durations(ds_EB64_fac_segs)
wavelet_duration_max_sec, wavelet_f_min_hz, WAVELET_J = choose_wavelet_j_max(
    wavelet_segment_durations_sec,
    fs=WAVELET_FS_HZ,
    dt=WAVELET_DT_SEC,
    s0=WAVELET_S0,
    dj=WAVELET_DJ,
    target_f_min=WAVELET_TARGET_F_MIN_HZ,
    min_cycles=WAVELET_MIN_CYCLES,
)

print('max segment length [s] =', wavelet_duration_max_sec)
print('f_min_seg [Hz] =', wavelet_f_min_hz)
print('J =', WAVELET_J)

wavelet_expected_paths, wavelet_all_outputs_exist = existing_wavelet_segment_paths(ds_EB64_fac_segs)
SKIP_WAVELET_CALIBRATION = (not WAVELET_OVERWRITE) and wavelet_all_outputs_exist

if SKIP_WAVELET_CALIBRATION:
    wavelet_ds_KS = None
    wavelet_calibration_df = pd.DataFrame()
    print('skip build_poynting_calibration_curve: existing wavelet segment files found and WAVELET_OVERWRITE=False')
    print(wavelet_expected_paths)
else:
    wavelet_ds_KS, wavelet_calibration_df = tw.build_poynting_calibration_curve(
        tw=tw,
        fs=WAVELET_FS_HZ,
        duration=wavelet_duration_max_sec,
        E0=10.0,
        B0=2.0,
        phase_deg=0.0,
        s0=WAVELET_S0,
        dj=WAVELET_DJ,
        J=WAVELET_J,
        use_coi_mask=True,
        trim_cycles=5,
    )

    print(wavelet_calibration_df[[
        'f0_input_hz', 'f_bin_hz', 'scale_bin',
        'K_signed_for_mean_flux', 'K_abs_for_absflux',
    ]])


In [ ]:
WAVELET_INPUT_VARS_64 = [
    'E64_fac_x', 'E64_fac_y', 'E64_fac_z',
    'B64_fac_x', 'B64_fac_y', 'B64_fac_z',
]


def close_loaded_wavelet_datasets():
    import gc
    for ds in globals().get('ds_EB64_fac_cwt_segs', []):
        try:
            ds.close()
        except Exception:
            pass
    globals().pop('ds_EB64_fac_cwt_segs', None)
    try:
        xr.backends.file_manager.FILE_CACHE.clear()
    except Exception:
        pass
    gc.collect()


def save_wavelet_dataset_atomic(ds, out_path, vars_to_save):
    tmp_path = out_path.with_name(f"{out_path.stem}.tmp{out_path.suffix}")
    if tmp_path.exists():
        tmp_path.unlink()
    ds[vars_to_save].to_netcdf(tmp_path)
    ds.close()
    try:
        tmp_path.replace(out_path)
    except PermissionError as exc:
        tmp_path.unlink(missing_ok=True)
        raise PermissionError(
            f"Cannot replace {out_path}. The file is probably still open in this kernel. "
            "Run close_loaded_wavelet_datasets(), or restart the kernel if the lock remains."
        ) from exc


def compute_wavelet_segment(ds_seg, i_seg, *, overwrite=False):
    out_path = wavelet_segment_path(i_seg)
    if out_path.exists() and not overwrite:
        print(f'[{i_seg+1}/{len(ds_EB64_fac_segs)}] keep existing: {out_path.name}')
        return out_path

    ds_cwt = tw.cwt_from_dataset(
        ds_seg,
        dt=WAVELET_DT_SEC,
        s0=WAVELET_S0,
        dj=WAVELET_DJ,
        J=WAVELET_J,
        variables=WAVELET_INPUT_VARS_64,
        auto_calibrate_psd=False,
        calibration_duration=wavelet_duration_max_sec,
        calibration_A0=1.0,
        e_unit='mV/m',
        b_unit='nT',
        progress=True,
        progress_desc=f'CWT seg {i_seg}',
    )
    if not ds_cwt.dims:
        print(f'skip seg_{i_seg}: empty CWT result')
        return None

    wco_exby, phase_exby = tw.calculate_xwt_wco(
        ds_cwt, 'E64_fac_x_coef', 'B64_fac_y_coef', WAVELET_DT_SEC, WAVELET_DJ
    )
    wco_eybx, phase_eybx = tw.calculate_xwt_wco(
        ds_cwt, 'E64_fac_y_coef', 'B64_fac_x_coef', WAVELET_DT_SEC, WAVELET_DJ
    )

    we_x = ds_cwt['E64_fac_x_coef'].values.astype(np.complex64)
    we_y = ds_cwt['E64_fac_y_coef'].values.astype(np.complex64)
    wb_x = ds_cwt['B64_fac_x_coef'].values.astype(np.complex64)
    wb_y = ds_cwt['B64_fac_y_coef'].values.astype(np.complex64)
    freqs = ds_cwt['freq'].values.astype(np.float64)
    scales = tw._get_scale_from_freq(freqs, WAVELET_DT_SEC).astype(np.float32)
    k_signed_f = wavelet_ds_KS['K_signed'].interp(freq=xr.DataArray(freqs, dims='freq')).values.astype(np.float32)

    cross_exby = np.real(we_x * np.conj(wb_y))
    cross_eybx = -np.real(we_y * np.conj(wb_x))
    spara_exby = k_signed_f[None, :] * cross_exby / scales[None, :] * 1e-12
    spara_eybx = k_signed_f[None, :] * cross_eybx / scales[None, :] * 1e-12

    ds_cwt['EB64_wco_exby'] = (('time', 'freq'), wco_exby.astype(np.float32))
    ds_cwt['EB64_phase_exby'] = (('time', 'freq'), phase_exby.astype(np.float32))
    ds_cwt['EB64_wco_eybx'] = (('time', 'freq'), wco_eybx.astype(np.float32))
    ds_cwt['EB64_phase_eybx'] = (('time', 'freq'), phase_eybx.astype(np.float32))
    ds_cwt['EB64_Spara_exby'] = (('time', 'freq'), spara_exby.astype(np.float32))
    ds_cwt['EB64_Spara_eybx'] = (('time', 'freq'), spara_eybx.astype(np.float32))

    vars_to_save = [name for name in ds_cwt.data_vars if isinstance(name, str) and not name.endswith('_coef')]
    save_wavelet_dataset_atomic(ds_cwt, out_path, vars_to_save)
    print(f'[{i_seg+1}/{len(ds_EB64_fac_segs)}] saved: {out_path.name}')
    return out_path


def run_wavelet_segments(ds_segs, *, overwrite=False, n_jobs=1):
    if overwrite:
        close_loaded_wavelet_datasets()
    tasks = list(enumerate(ds_segs))
    paths = []
    if n_jobs <= 1 or len(tasks) <= 1:
        for i_seg, ds_seg in tqdm(tasks, desc='Wavelet segments', unit='seg'):
            path = compute_wavelet_segment(ds_seg, i_seg, overwrite=overwrite)
            if path is not None:
                paths.append(path)
        return paths

    print(f'Running wavelet segments with {n_jobs} worker threads')
    with cf.ThreadPoolExecutor(max_workers=n_jobs) as executor:
        future_map = {
            executor.submit(compute_wavelet_segment, ds_seg, i_seg, overwrite=overwrite): i_seg
            for i_seg, ds_seg in tasks
        }
        for fut in tqdm(cf.as_completed(future_map), total=len(future_map), desc='Wavelet segments', unit='seg'):
            i_seg = future_map[fut]
            try:
                path = fut.result()
            except Exception as exc:
                raise RuntimeError(f'Wavelet segment {i_seg} failed') from exc
            if path is not None:
                paths.append(path)
    return sorted(paths, key=lambda path: int(str(path).split('_seg')[-1].split('.')[0]))


wavelet_segment_paths = run_wavelet_segments(
    ds_EB64_fac_segs,
    overwrite=WAVELET_OVERWRITE,
    n_jobs=WAVELET_N_JOBS,
)

if not wavelet_segment_paths:
    raise ValueError('No wavelet segment files were produced or found')


In [ ]:
def concat_cwt_segments(dsets, var):
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError('dsets must be a Dataset or a list of Datasets')

    das = [ds[var] for ds in dsets if isinstance(ds, xr.Dataset) and var in ds.data_vars]
    if not das:
        return None, None

    pow_cat = xr.concat(das, dim='time').sortby('time')
    coi_name = var.replace('_cwt', '_coi')
    coi_list = [ds[coi_name] for ds in dsets if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars]
    coi_cat = xr.concat(coi_list, dim='time').sortby('time') if coi_list else None
    return pow_cat, coi_cat


def concat_xwt_segments(dsets, var):
    das = [ds[var] for ds in dsets if isinstance(ds, xr.Dataset) and var in ds.data_vars]
    if not das:
        return None
    return xr.concat(das, dim='time').sortby('time')


def _time_slice_2d(da, t0=None, t1=None, minutes=5):
    if t0 is None:
        return da, t1
    if t1 is None:
        t1 = np.datetime64(t0) + np.timedelta64(minutes, 'm')
    return da.sel(time=slice(t0, t1)), t1


def _format_window_time(t0):
    return str(np.datetime64(t0)).replace(':', '').replace('T', '_')


def _thin_time_for_plot(da, max_time_points=None):
    if da is None:
        return None
    if max_time_points is None:
        max_time_points = WAVELET_PLOT_MAX_TIME_POINTS
    n_time = da.sizes.get('time', 0)
    if n_time <= max_time_points:
        return da
    stride = int(np.ceil(n_time / max_time_points))
    return da.isel(time=slice(None, None, stride))


def cleanup_plot_memory(fig=None):
    if fig is not None:
        fig.clf()
        plt.close(fig)
    plt.close('all')
    gc.collect()


def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, t1=None, minutes=5,
                   f_range=(1e-2, 64.0), zrange=(1e-6, 1e3), cmap='turbo',
                   ylabel='', unit_right='', cax=None):
    da, t1 = _time_slice_2d(da_pow, t0=t0, t1=t1, minutes=minutes)
    coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None and t0 is not None else da_coi
    da = _thin_time_for_plot(da)
    coi = _thin_time_for_plot(coi) if coi is not None else None
    if da.sizes.get('time', 0) == 0:
        return None, None

    time_num = mdates.date2num(da.time.values)
    freq = da.freq.values
    z = np.asarray(da.values, dtype=np.float32)
    if coi is not None:
        z = np.where(freq[None, :] < coi.values[:, None], np.nan, z).astype(np.float32, copy=False)

    pcm = ax.pcolormesh(time_num, freq, z.T, shading='auto', norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap, rasterized=True)
    ax.set_yscale('log')
    ax.set_ylim(*f_range)
    ax.set_ylabel(f'{ylabel}\n[Hz]')
    if t0 is not None:
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype='datetime64[ns]')))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.minorticks_on()
    ax.grid(which='both', alpha=0.3)
    if cax is None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = ax.figure.colorbar(pcm, cax=cax)
    cb.set_label(unit_right)
    return pcm, cb


def plot_xwt_phase_on_ax(ax, da_wco, da_phase, t0=None, t1=None, minutes=5,
                         wco_thresh=0.0, yrange=(1e-2, 4.0), mode='wco',
                         label_left='', unit=None, cax=None):
    wco, t1 = _time_slice_2d(da_wco, t0=t0, t1=t1, minutes=minutes)
    phase = da_phase.sel(time=slice(t0, t1)) if t0 is not None else da_phase
    wco = _thin_time_for_plot(wco)
    phase = _thin_time_for_plot(phase)
    if wco.sizes.get('time', 0) == 0:
        return None, None

    time_num = mdates.date2num(wco.time.values)
    freq = wco.freq.values
    if mode == 'wco':
        z = np.asarray(wco.values, dtype=np.float32)
        cmap = 'viridis'
        norm = Normalize(vmin=0, vmax=1)
        unit = 'Coherency' if unit is None else unit
    else:
        z = np.abs(phase.where(wco > wco_thresh).values.astype(np.float32))
        cmap = 'Spectral'
        norm = Normalize(vmin=0, vmax=180)
        unit = 'Phase (abs) [deg]' if unit is None else unit

    pcm = ax.pcolormesh(time_num, freq, z.T, shading='auto', norm=norm, cmap=cmap, rasterized=True)
    ax.set_yscale('log')
    ax.set_ylim(*yrange)
    ax.set_ylabel(f'{label_left}\n[Hz]')
    if t0 is not None:
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype='datetime64[ns]')))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.minorticks_on()
    ax.grid(which='both', alpha=0.3)
    if cax is None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = ax.figure.colorbar(pcm, cax=cax)
    cb.set_label(unit)
    if mode == 'phase':
        cb.set_ticks([0, 45, 90, 135, 180])
    return pcm, cb


def plot_Spara_on_ax(ax, da, da_coi=None, t0=None, t1=None, minutes=5,
                     f_range=(1e-2, 64.0), zrange=(-1e-3, 1e-3), cmap='turbo',
                     ylabel='', unit_right='', cax=None, linthresh=1e-6):
    da, t1 = _time_slice_2d(da, t0=t0, t1=t1, minutes=minutes)
    coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None and t0 is not None else da_coi
    da = _thin_time_for_plot(da)
    coi = _thin_time_for_plot(coi) if coi is not None else None
    if da.sizes.get('time', 0) == 0:
        return None, None

    time_num = mdates.date2num(da.time.values)
    freq = da.freq.values
    z = np.asarray(da.values, dtype=np.float32)
    if coi is not None:
        z = np.where(freq[None, :] < coi.values[:, None], np.nan, z).astype(np.float32, copy=False)

    finite = np.isfinite(z)
    if not finite.any():
        ax.text(0.5, 0.5, 'No finite data in this window', ha='center', va='center', transform=ax.transAxes)
        ax.set_yscale('log')
        ax.set_ylim(*f_range)
        ax.set_ylabel(f'{ylabel}\n[Hz]')
        if t0 is not None:
            ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype='datetime64[ns]')))
        return None, None

    vmin, vmax = zrange
    if not (np.isfinite(vmin) and np.isfinite(vmax)) or vmin >= vmax:
        vmax = np.nanmax(np.abs(z[finite]))
        vmin = -vmax
    norm = SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)

    tm, fm = np.meshgrid(time_num, freq, indexing='ij')
    pcm = ax.pcolormesh(time_num, freq, z.T, shading='auto', norm=norm, cmap=cmap, rasterized=True)
    ax.set_yscale('log')
    ax.set_ylim(*f_range)
    ax.set_ylabel(f'{ylabel}\n[Hz]')
    if t0 is not None:
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype='datetime64[ns]')))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.minorticks_on()
    ax.grid(which='both', alpha=0.3)
    if cax is None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = ax.figure.colorbar(pcm, cax=cax)
    cb.set_label(unit_right)
    return pcm, cb


In [ ]:
targets_EB64_fac_cwt = [
    ('E64_fac_x_cwt', r'$E_{x}$ (FAC)', '[(mV/m)$^2$/Hz]'),
    ('E64_fac_y_cwt', r'$E_{y}$ (FAC)', '[(mV/m)$^2$/Hz]'),
    ('E64_fac_z_cwt', r'$E_{z}$ (FAC)', '[(mV/m)$^2$/Hz]'),
    ('B64_fac_x_cwt', r'$B_{x}$ (FAC)', '[nT$^2$/Hz]'),
    ('B64_fac_y_cwt', r'$B_{y}$ (FAC)', '[nT$^2$/Hz]'),
    ('B64_fac_z_cwt', r'$B_{z}$ (FAC)', '[nT$^2$/Hz]'),
]

targets_EB64_xwt = [
    ('EB64_wco_exby', 'EB64_phase_exby', r'$E_{x}$ & $B_{y}$'),
    ('EB64_wco_eybx', 'EB64_phase_eybx', r'$E_{y}$ & $B_{x}$'),
]

targets_EB64_fac_Spara = [
    ('EB64_Spara_exby', r'$S_{\parallel}^{E_{x} B_{y}}$ (FAC)', '[W/m$^2$]'),
    ('EB64_Spara_eybx', r'$S_{\parallel}^{E_{y} B_{x}}$ (FAC)', '[W/m$^2$]'),
]

missing_wavelet_files = [path for path in wavelet_segment_paths if not Path(path).exists()]
if missing_wavelet_files:
    raise FileNotFoundError(f'Missing wavelet spectra files: {missing_wavelet_files}')

ds_EB64_fac_cwt_segs = [xr.open_dataset(path) for path in wavelet_segment_paths]
print(ds_EB64_fac_cwt_segs)

joined_EB64_fac_cwt = {}
for var_name, _, _ in targets_EB64_fac_cwt:
    da, coi = concat_cwt_segments(ds_EB64_fac_cwt_segs, var_name)
    if da is not None:
        joined_EB64_fac_cwt[var_name] = (da, coi)

joined_EB64_xwt = {}
for w_var, p_var, label in targets_EB64_xwt:
    w_da = concat_xwt_segments(ds_EB64_fac_cwt_segs, w_var)
    p_da = concat_xwt_segments(ds_EB64_fac_cwt_segs, p_var)
    if w_da is not None and p_da is not None:
        joined_EB64_xwt[label] = (w_da, p_da)

joined_EB64_fac_Spara = {}
for var_name, _, _ in targets_EB64_fac_Spara:
    da = concat_xwt_segments(ds_EB64_fac_cwt_segs, var_name)
    if da is not None:
        joined_EB64_fac_Spara[var_name] = da

# Compatibility names used by later exploratory cells.
da_E64_fac_x_cwt = joined_EB64_fac_cwt['E64_fac_x_cwt']
da_E64_fac_y_cwt = joined_EB64_fac_cwt['E64_fac_y_cwt']
da_E64_fac_z_cwt = joined_EB64_fac_cwt['E64_fac_z_cwt']
da_B64_fac_x_cwt = joined_EB64_fac_cwt['B64_fac_x_cwt']
da_B64_fac_y_cwt = joined_EB64_fac_cwt['B64_fac_y_cwt']
da_B64_fac_z_cwt = joined_EB64_fac_cwt['B64_fac_z_cwt']


In [ ]:
def save_wavelet_cwt_panels(time_windows, out_dir=wavelet_plot_dir / 'wavelet_cwt'):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for t0 in tqdm(time_windows, desc='CWT panel plots', unit='win'):
        fig, axes = plt.subplots(len(targets_EB64_fac_cwt), 1, figsize=(10, 10), sharex=True)
        for ax, (var_name, ylab, unit) in zip(axes, targets_EB64_fac_cwt):
            if var_name not in joined_EB64_fac_cwt:
                continue
            da, coi = joined_EB64_fac_cwt[var_name]
            plot_cwt_on_ax(
                ax, da, coi, t0=t0, minutes=5,
                zrange=(1e-6, 1e3), f_range=(1e-2, 32.0),
                cmap='turbo', ylabel=ylab, unit_right=unit,
            )
        axes[-1].set_xlabel('time')
        fig.tight_layout()
        fig_path = out_dir / f"EB_fields_fac_cwt_{_format_window_time(t0)}.png"
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        cleanup_plot_memory(fig)


def save_wavelet_xwt_panels(time_windows, out_dir=wavelet_plot_dir / 'wavelet_xwt', wco_thresh=0.5):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for t0 in tqdm(time_windows, desc='XWT/phase panel plots', unit='win'):
        fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
        for i, (w_var, p_var, label) in enumerate(targets_EB64_xwt):
            if label not in joined_EB64_xwt:
                continue
            w_da, p_da = joined_EB64_xwt[label]
            yrange = (float(np.nanmin(p_da.freq)), float(np.nanmax(p_da.freq)))
            plot_xwt_phase_on_ax(
                axes[2*i], w_da, p_da, t0=t0, mode='wco',
                label_left=f'Coherence ({label})', yrange=yrange,
            )
            plot_xwt_phase_on_ax(
                axes[2*i+1], w_da, p_da, t0=t0, mode='phase', wco_thresh=wco_thresh,
                label_left=f'Phase ({label})', yrange=yrange,
            )
        axes[-1].set_xlabel('time')
        fig.tight_layout()
        fig_path = out_dir / f"EB_fields_fac_64_xwt_{_format_window_time(t0)}_wco{wco_thresh:g}.png"
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        cleanup_plot_memory(fig)


def save_wavelet_spara_panels(time_windows, out_dir=wavelet_plot_dir / 'wavelet_Spara'):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for t0 in tqdm(time_windows, desc='Spara wavelet plots', unit='win'):
        fig, axes = plt.subplots(len(targets_EB64_fac_Spara), 1, figsize=(10, 5), sharex=True)
        for ax, (var_name, ylab, _) in zip(axes, targets_EB64_fac_Spara):
            if var_name not in joined_EB64_fac_Spara:
                continue
            da = joined_EB64_fac_Spara[var_name]
            plot_Spara_on_ax(
                ax, da * 1e3, t0=t0, minutes=5,
                zrange=(-1e-2, 1e-2),
                f_range=(float(np.nanmin(da.freq)), float(np.nanmax(da.freq))),
                cmap='BrBG', ylabel=ylab, unit_right='[mW/m$^2$]', linthresh=1e-5,
            )
        axes[-1].set_xlabel('time')
        fig.tight_layout()
        fig_path = out_dir / f"EB_fields_fac_Spara_{_format_window_time(t0)}.png"
        print(fig_path)
        fig.savefig(fig_path, bbox_inches='tight')
        cleanup_plot_memory(fig)


if PLOT_WAVELET_DIAGNOSTICS:
    if PLOT_WAVELET_CWT_PANELS:
        save_wavelet_cwt_panels(time_windows_5min)
    if PLOT_WAVELET_XWT_PANELS:
        save_wavelet_xwt_panels(time_windows_5min)
    if PLOT_WAVELET_SPARA_PANELS:
        save_wavelet_spara_panels(time_windows_5min)


# 5分毎のPSDのMedianをplot


In [ ]:
def _unwrap_cwt_power(da_or_pair):
    return da_or_pair[0] if isinstance(da_or_pair, tuple) else da_or_pair


def compute_median_psd_dict(pairs_sorted, t0, t1):
    medians = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = sub.real**2 + sub.imag**2
        medians[name] = sub.median(dim='time', skipna=True)
    return medians


def plot_median_psd_dict(medians, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    color_map = {
        ('E', 'x'): 'blue',
        ('E', 'y'): 'orange',
        ('E', 'z'): 'brown',
        ('B', 'x'): 'green',
        ('B', 'y'): 'red',
        ('B', 'z'): 'purple',
    }
    for name, med in medians.items():
        parts = name.split('_')
        prefix, coord, comp = parts[0], parts[2], parts[3]
        if comp == 'z':
            continue
        ax.loglog(med['freq'], med, label=f'${prefix}_{comp}$ ({coord})', lw=1, color=color_map.get((prefix, comp), 'gray'))

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:]}-{str(t1)[11:]}\n(E: (mV/m)$^2$/Hz, B: nT$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4])
    ax.legend(ncol=2, fontsize=15)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 1e4)
    fig.tight_layout()

    if outdir is None:
        plt.show()
    else:
        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)
        fig_path = outdir / f"median_psd_EB_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    cleanup_plot_memory(fig)


def save_5min_median_psd(time_windows, outdir=wavelet_plot_dir / '5min_PSD'):
    pairs = [
        ('E_64_FAC_x_cwt', _unwrap_cwt_power(da_E64_fac_x_cwt)),
        ('E_64_FAC_y_cwt', _unwrap_cwt_power(da_E64_fac_y_cwt)),
        ('E_64_FAC_z_cwt', _unwrap_cwt_power(da_E64_fac_z_cwt)),
        ('B_64_FAC_x_cwt', _unwrap_cwt_power(da_B64_fac_x_cwt)),
        ('B_64_FAC_y_cwt', _unwrap_cwt_power(da_B64_fac_y_cwt)),
        ('B_64_FAC_z_cwt', _unwrap_cwt_power(da_B64_fac_z_cwt)),
    ]
    pairs_sorted = [(name, da.sortby('time')) for name, da in pairs if isinstance(da, xr.DataArray)]
    for t0 in tqdm(time_windows, desc='5min median PSD', unit='win'):
        t1 = t0 + np.timedelta64(5, 'm')
        medians = compute_median_psd_dict(pairs_sorted, t0, t1)
        if medians:
            plot_median_psd_dict(medians, t0, t1, outdir=outdir)


if PLOT_WAVELET_DIAGNOSTICS and PLOT_WAVELET_MEDIAN_PSD:
    save_5min_median_psd(time_windows_5min)


In [ ]:
def concat_fac_component(ds_segs, var_name):
    return xr.concat([ds_seg[var_name] for ds_seg in ds_segs if var_name in ds_seg], dim='time').sortby('time')


mu_0 = 4.0 * np.pi * 1e-7
Ex_fac = concat_fac_component(ds_EB64_fac_segs, 'E64_fac_x')
Ey_fac = concat_fac_component(ds_EB64_fac_segs, 'E64_fac_y')
Bx_fac = concat_fac_component(ds_EB64_fac_segs, 'B64_fac_x')
By_fac = concat_fac_component(ds_EB64_fac_segs, 'B64_fac_y')

S_para_toroidal = Ex_fac * By_fac / mu_0 * 1e-12
S_para_poloidal = -Ey_fac * Bx_fac / mu_0 * 1e-12
S_para = S_para_toroidal + S_para_poloidal

Vph_toroidal = np.abs(Ey_fac / Bx_fac) * 1e6
Vph_poloidal = np.abs(Ex_fac / By_fac) * 1e6
Vph_perp_comp = np.sqrt((Ex_fac**2 + Ey_fac**2) / (Bx_fac**2 + By_fac**2)) * 1e6

print(S_para)
print(S_para_toroidal)
print(S_para_poloidal)


In [ ]:
def add_panel_label(ax, label, x=-0.075, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes, ha='right', va='bottom', clip_on=False)


def plot_spara_timeseries(out_dir=wavelet_plot_dir):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(S_para.time, np.abs(S_para.values * 1e3), s=0.1, c='k')
    ax.set_yscale('log')
    ax.set_xlabel('Time')
    ax.set_ylabel(r'$|S_{\parallel}|$ [mW/m$^{2}$]')
    ax.set_ylim(bottom=1e-7, top=1e0)
    ax.set_xlim(np.datetime64(t_start), np.datetime64(t_end))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.minorticks_on()
    ax.grid(which='both', alpha=0.5)
    ax.axhline(y=0.009, c='r', linestyle='--', alpha=0.5)
    add_panel_label(ax, '(b)')
    fig.tight_layout()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ('png', 'pdf'):
        fig_path = out_dir / f'Arase_S_para.{suffix}'
        print(fig_path)
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


if PLOT_WAVELET_DIAGNOSTICS:
    plot_spara_timeseries()


# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import xarray as xr

EB_ANALYSIS_COMPONENT = "toroidal"
BUILD_EB_COMPONENTS = [EB_ANALYSIS_COMPONENT]

EB_COMPONENT_CONFIGS = {
    "toroidal": {
        "name": "toroidal",
        "title_label": r"$E_{x}$-$B_{y}$",
        "E_var": "E64_fac_x_cwt",
        "B_var": "B64_fac_y_cwt",
        "coherency_var": "EB64_wco_exby",
        "phase_var": "EB64_phase_exby",
        "Spara_cwt_var": "EB64_Spara_exby",
        "Spara_source": "S_para_toroidal",
        "velocity_source": "ds_velocity_ms_toroidal",
        "E2_label": r"$E_{x}^{2}$",
        "B2_label": r"$B_{y}^{2}$",
        "phi_label": r"$\phi$",
        "EBratio_label": r"$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$",
        "vsys_label": r"$V_{\mathrm{sys}x}$",
        "Spara_label": r"$S_{\parallel}^{E_{x} B_{y}}$",
        "S_par_label": r"$S_{\parallel}^{E_{x} B_{y}}$",
    },
    # 後で poloidal を実装する場合は BUILD_EB_COMPONENTS に "poloidal" を追加する。
    "poloidal": {
        "name": "poloidal",
        "title_label": r"$E_{y}$-$B_{x}$",
        "E_var": "E64_fac_y_cwt",
        "B_var": "B64_fac_x_cwt",
        "coherency_var": "EB64_wco_eybx",
        "phase_var": "EB64_phase_eybx",
        "Spara_cwt_var": "EB64_Spara_eybx",
        "Spara_source": "S_para_poloidal",
        "velocity_source": "ds_velocity_ms_poloidal",
        "E2_label": r"$E_{y}^{2}$",
        "B2_label": r"$B_{x}^{2}$",
        "phi_label": r"$\phi$",
        "EBratio_label": r"$\sqrt{E_{y}^{2} / B_{x}^{2}} / v_{\mathrm{A}}$",
        "vsys_label": r"$V_{\mathrm{sys}y}$",
        "Spara_label": r"$S_{\parallel}^{E_{y} B_{x}}$",
        "S_par_label": r"$S_{\parallel}^{E_{y} B_{x}}$",
    },
}

eb_cfg = EB_COMPONENT_CONFIGS[EB_ANALYSIS_COMPONENT]

def eb_output_dir(component_name=EB_ANALYSIS_COMPONENT):
    return Path(
        "/mnt/j/statistical_analysis_arase/preanalysis/KAW_observation/E_B_ratio_Arase"
    ) / folder_time_label / f"EB_ratio_{component_name}_eachtime"


In [ ]:
def make_wco_sig95_segments(ds_segs, coherency_var, min_value=0.5):
    return [
        (ds_seg[coherency_var] * 0.0).clip(min=min_value)
        for ds_seg in ds_segs
    ]


In [ ]:
def concat_seg_var(ds_segs, var_name):
    return xr.concat(
        [ds_seg[var_name] for ds_seg in ds_segs],
        dim="time",
    )


def build_eb_component_dataset(ds_segs, cfg):
    wco_sig95 = xr.concat(
        make_wco_sig95_segments(ds_segs, cfg["coherency_var"]),
        dim="time",
    )
    return xr.Dataset({
        "E64":       concat_seg_var(ds_segs, cfg["E_var"]),
        "B64":       concat_seg_var(ds_segs, cfg["B_var"]),
        "coherency": concat_seg_var(ds_segs, cfg["coherency_var"]),
        "phase":     concat_seg_var(ds_segs, cfg["phase_var"]),
        "wco_sig95": wco_sig95,
        "Spara":     concat_seg_var(ds_segs, cfg["Spara_cwt_var"]),
    }).sortby("time")


ds_EB64_fac_components = {
    name: build_eb_component_dataset(ds_EB64_fac_cwt_segs, EB_COMPONENT_CONFIGS[name])
    for name in BUILD_EB_COMPONENTS
}

ds_EB64_fac_analysis = ds_EB64_fac_components[EB_ANALYSIS_COMPONENT]

# 既存の下流セルとの互換性を残す。現時点では toroidal のみ実体化する。
ds_EB64_fac_toroidal = ds_EB64_fac_analysis


## 各時間で出力してみる

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

new_time = pd.DatetimeIndex(ds_EB64_fac_analysis.time.values)

Spara_source = globals()[eb_cfg["Spara_source"]]
velocity_source = globals()[eb_cfg["velocity_source"]]

ds_EB64_fac_analysis_interp = ds_EB64_fac_analysis.interp(
    time=new_time,
    method="linear",
).assign_coords(time=new_time)

da_Spara_analysis_interp = Spara_source.interp(
    time=new_time,
    method="linear",
).assign_coords(time=new_time)

ds_parameter_interp = ds_parameter.interp(
    time=new_time,
    method="linear",
).assign_coords(time=new_time)

ds_velocity_ms_analysis_interp = velocity_source.interp(
    time=new_time,
    method="linear",
).assign_coords(time=new_time)

# 既存の関数・セルが参照する名前を toroidal 実行用 alias として残す。
ds_EB64_fac_toroidal_interp = ds_EB64_fac_analysis_interp
da_Spara_toroidal_interp = da_Spara_analysis_interp
ds_velocity_ms_toroidal_interp = ds_velocity_ms_analysis_interp


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq)
    y = np.asarray(psd)

    msk = np.asarray(mask, dtype=bool)
    msk &= np.isfinite(f) & np.isfinite(y) & (f > 0) & (y > 0)

    if msk.sum() < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    N = len(x)

    # 線形回帰
    slope, intercept = np.polyfit(x, yy, 1)

    # 残差
    y_fit = intercept + slope * x
    residual = yy - y_fit

    # 残差分散
    sigma2 = np.sum(residual**2) / (N - 2)

    Sxx = np.sum((x - x.mean())**2)

    if Sxx == 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / Sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std


def _corr_log_model(freq, y_obs, y_model_at_freq, min_points=5):
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model_at_freq)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    if msk.sum() < min_points:
        return np.nan, int(msk.sum())

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    # 分散が小さいと相関が不安定なので弾く
    if np.std(logy) < 1e-6 or np.std(logm) < 1e-6:
        return np.nan, int(msk.sum())

    r = np.corrcoef(logy, logm)[0, 1]
    return float(r), int(msk.sum())

def _logrmse_model(freq, y_obs, y_model, min_points=5, allow_offset=False):
    """
    logRMSE = sqrt(mean((log10(y_obs) - (log10(y_model)+a))^2))
    allow_offset=True: a を平均差で最小二乗フィット（縦オフセット許容）
    """
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    n = int(msk.sum())
    if n < min_points:
        return np.nan, np.nan, n  # (logRMSE, a, n)

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    a = 0.0
    if allow_offset:
        a = float(np.mean(logy - logm))
        resid = logy - (logm + a)
    else:
        resid = logy - logm

    rmse = float(np.sqrt(np.mean(resid**2)))
    return rmse, a, n



def plot_freq_spectrum(time, ds_64, ds_par, ds_vel, da_Spara, title_label,
                      E2_label, B2_label, phi_label, EBratio_label, vsys_label, S_para_label,
                      fit_range=(3.0, 30.0), fig_plot=True, save_dir=None):

    time = pd.Timestamp(time)
    fit_range = list(fit_range)

    ds_64_time      = ds_64.sel(time=time, method="nearest")
    ds_par_time     = ds_par.sel(time=time, method="nearest")
    ds_vel_time     = ds_vel.sel(time=time, method="nearest")
    da_Spara_time   = da_Spara.sel(time=time, method="nearest")

    # 理論曲線
    f_sc    = np.logspace(-2, 2, 1000)
    tau     = ds_par_time['i-e_temp_ratio'].item()
    f_ci    = ds_par_time['proton_cycl_freq_Hz'].item() * proton_mass_kg / ds_par_time['ion_mass_kg'].item()
    v_thi   = ds_vel_time['ion_thermal_speed'].item()
    v_sys   = ds_vel_time['perp_sys_speed'].item()
    S_para  = da_Spara_time.item()
    KAW_dr  = (1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    if fit_range[1] == 0:
        fit_range[1] = np.sqrt(ds_par_time['ion_mass_kg'].item() / 9.1093837E-31 * tau)

    f_sc_krho_1     = np.abs(f_ci * v_sys / v_thi)
    f_sc_krho_low   = f_sc_krho_1 * fit_range[0]
    f_sc_krho_high  = f_sc_krho_1 * fit_range[1]

    f_spin_1 = 0.125
    f_spin_2 = f_spin_1 * 2.
    f_spin_3 = f_spin_1 * 3.
    f_spin_4 = f_spin_1 * 4.
    f_spin_5 = f_spin_1 * 5.

    # PSD
    E_64        = ds_64_time['E64']
    B_64        = ds_64_time['B64']

    # --- E64 amplitude fraction in 3-32 Hz ---
    E64_threshold = 1e-4

    E64_band = E_64.sel(freq=slice(32.0, 3.0))
    E64_band_values = E64_band.values

    E64_valid = np.isfinite(E64_band_values)
    E64_n_total_3_32 = int(E64_valid.sum())
    E64_n_over_1e4_3_32 = int((E64_valid & (E64_band_values >= E64_threshold)).sum())

    if E64_n_total_3_32 > 0:
        E64_frac_over_1e4_3_32 = E64_n_over_1e4_3_32 / E64_n_total_3_32
    else:
        E64_frac_over_1e4_3_32 = np.nan

    coherency   = ds_64_time['coherency']
    wco_sig95   = ds_64_time['wco_sig95']
    phase       = ds_64_time['phase']

    v_A     = ds_vel_time['Alfven_speed_MID'].item()

    EB_64_ratio     = np.sqrt(E_64 / B_64) * 1E6 / v_A

    # --- fit用マスク（coherency + fit_range） ---
    freq = E_64.freq.values
    mask_coh = (coherency >= wco_sig95).values
    mask_fit = (freq >= f_sc_krho_low) & (freq <= f_sc_krho_high) & (freq >= 1E-2)
    mask_all = mask_coh & mask_fit

    KAW_dr_corr = (1. + (freq / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (freq / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    for_max_n_EB_dr     = KAW_dr_corr[mask_fit]
    for_max_n_EB_ratio  = EB_64_ratio[mask_fit]

    KAW_dr_corr = KAW_dr_corr[mask_all]
    EB_64_ratio_corr = EB_64_ratio[mask_all]
    EB_64_ratio_coh     = EB_64_ratio.where(coherency >= wco_sig95)

    phase_coh   = phase.where(coherency >= wco_sig95)

    _, n_EB_max             = _corr_log_model(for_max_n_EB_ratio.freq, for_max_n_EB_ratio.data, for_max_n_EB_dr, min_points=10)

    r_EB, n_EB              = _corr_log_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)
    logrmse_EB_abs, _, _    = _logrmse_model(EB_64_ratio_corr.freq, EB_64_ratio_corr.data, KAW_dr_corr, min_points=10)

    # κ推定（失敗なら NaN）
    kappa_E, kappa_E_err    = _fit_powerlaw_kappa(freq, E_64.values, mask_all, min_points=10)
    kappa_B, kappa_B_err    = _fit_powerlaw_kappa(freq, B_64.values, mask_all, min_points=10)

    # Spara (frequency依存)
    Spara_freq_64   = ds_64_time['Spara'] * 1E3 # [mW m-2]
    Spara_freq_64   = Spara_freq_64.where(coherency >= wco_sig95)

    #v_g_para_64     = v_A * np.sqrt(1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2. * (1. + 1. / tau))   # [m s-1]

    #Effective_wave_energy_density_64    = np.abs(Spara_freq_64 / v_g_para_64) * 1E-3 / elementary_charge * 1E-6 # [eV cm-3]

    wave_energy_density_64  = 0.5 / mu_0 / v_A**2. / (1. + 0.5 * (Spara_freq_64.freq / f_ci * v_thi / v_sys)**2.) * E_64*1E-6 / elementary_charge * 1E-6   # [eV cm-3 Hz-1]

    wave_energy_density_64 = wave_energy_density_64.where(coherency >= wco_sig95)

    if fig_plot == True:
        # plot
        import matplotlib as mpl
        import matplotlib.pyplot as plt

        mpl.rcParams['font.size'] = 20

        fig     = plt.figure(figsize=(10, 20))
        gs      = fig.add_gridspec(6, 1, height_ratios=[4, 3, 3, 4, 4, 4], hspace=0.15)
        ax_0    = fig.add_subplot(gs[0, 0])
        ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
        ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
        ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)
        ax_4    = fig.add_subplot(gs[4, 0], sharex=ax_0)
        ax_5    = fig.add_subplot(gs[5, 0], sharex=ax_0)

        ax_0.tick_params(axis='x', which='both', labelbottom=False)
        ax_1.tick_params(axis='x', which='both', labelbottom=False)
        ax_2.tick_params(axis='x', which='both', labelbottom=False)
        ax_3.tick_params(axis='x', which='both', labelbottom=False)
        ax_4.tick_params(axis='x', which='both', labelbottom=False)

        # 灰色マスク（coherency>=sig95 だけ。fit_rangeではなく“灰色領域”を強調したいならこれ）
        mask_gray = mask_coh
        ax_0.plot(E_64.freq,   E_64,  lw=2, linestyle='solid',  c='green')
        ax_0.plot(B_64.freq,   B_64,  lw=2, linestyle='solid',  c='purple')
        ax_0.set_yscale('log')
        ax_0.set_ylim(1E-6, 1E4)
        ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
        ax_0.set_xscale('log')
        ax_0.set_xlim(1E-2, 64)
        ax_0.minorticks_on()
        ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_0.fill_between(freq, 1E-6, 1E4, where=mask_gray, color='gray', alpha=0.30)

        ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')

        ax_1.plot(wco_sig95.freq, wco_sig95, lw=2, linestyle='dotted', c='r')
        ax_1.plot(coherency.freq, coherency, lw=2, linestyle='solid',  c='k')
        #ax_1.set_yscale('log')
        #ax_1.set_ylim(1E-3, 1)
        ax_1.set_ylim(0, 1)
        ax_1.minorticks_on()
        ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_1.set_ylabel('Coherence')

        ax_2.plot(phase_coh.freq, np.abs(phase_coh), lw=2, linestyle='solid', c='k')
        ax_2.axhline(90, lw=2, linestyle='dashed', c='gray', alpha=0.5)
        ax_2.axhline(60, lw=2, linestyle='dashed', c='red', alpha=0.5)
        ax_2.axhline(120, lw=2, linestyle='dashed', c='red', alpha=0.5)
        ax_2.set_ylim(0, 180)
        ax_2.set_yticks([0, 45, 90, 135, 180])
        ax_2.minorticks_on()
        ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_2.set_ylabel('|' + phi_label + '| [deg]')

        ax_3.plot(EB_64_ratio_coh.freq,  EB_64_ratio_coh,  lw=2, linestyle='solid',  c='k')
        ax_3.plot(f_sc, KAW_dr, lw=2, linestyle='dotted', c='r')
        ax_3.set_yscale('log')
        ax_3.set_ylim(1E-1, 1E3)
        ax_3.minorticks_on()
        ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_3.set_ylabel(EBratio_label)
        if np.isfinite(r_EB):
            ax_3.text(
                0.98, 0.47, r'$r_{\mathrm{KAW}}$ =' + f'{r_EB:.2f}',
                transform=ax_3.transAxes, ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
            )
        if np.isfinite(logrmse_EB_abs):
            ax_3.text(
                0.98, 0.34, rf'logRMSE={logrmse_EB_abs:.2f}',
                transform=ax_3.transAxes, ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
            )
        ax_3.text(0.98, 0.21, r'$n_{\mathrm{KAW}}$' + f'= {n_EB}',
                  transform=ax_3.transAxes, ha='right', va='center',
                  bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))
        ax_3.text(0.98, 0.08, r'$n_{\mathrm{KAW, max}}$' + f'= {n_EB_max}',
                  transform=ax_3.transAxes, ha='right', va='center',
                  bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

        ax_4.plot(Spara_freq_64.freq, Spara_freq_64, lw=2, linestyle='solid', c='r')
        ax_4.plot(Spara_freq_64.freq, -Spara_freq_64, lw=2, linestyle='solid', c='b')
        ax_4.set_yscale('log')
        ax_4.set_ylim(1E-7, 1E-1)
        ax_4.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1])
        ax_4.minorticks_on()
        ax_4.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_4.set_ylabel(S_para_label + r' [$\mathrm{mW}/\mathrm{m}^{2}$]')

        W_para_label = S_para_label.replace('S_{\\parallel', 'W_').replace('}}$', '}$')

        #ax_5.plot(Effective_wave_energy_density_64.freq, Effective_wave_energy_density_64, lw=2, linestyle='solid', c='k')
        ax_5.plot(wave_energy_density_64.freq, wave_energy_density_64, lw=2, linestyle='solid', c='k')
        ax_5.set_yscale('log')
        ax_5.set_ylim(1E-7, 1E3)
        ax_5.set_yticks([1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3])
        ax_5.minorticks_on()
        ax_5.grid(which='both', alpha=0.3, linestyle='dashed')
        ax_5.set_ylabel(r'$W$ [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]')
        ax_5.set_xlabel('Frequency [Hz]')

        ax_0.set_title(
            f"{time.strftime('%H:%M:%S.%f')}" + '\n Arase, ' +
            S_para_label + f' = {(S_para*1E3):.3f} ' + r'[$\mathrm{mW/m^{2}}$]'
        )

        for ax in [ax_0, ax_1, ax_2, ax_3, ax_4, ax_5]:
            ax.axvline(f_spin_1, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_2, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_3, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_4, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_spin_5, lw=2, linestyle='dotted', c='blue', alpha=0.5)
            ax.axvline(f_sc_krho_1,   lw=2, linestyle='dashed', c='orange',  alpha=0.7)
            ax.axvline(f_sc_krho_low, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
            ax.axvline(f_sc_krho_high,lw=2, linestyle='dashed', c='magenta', alpha=0.7)

        def add_panel_label(ax, label, x=-0.15, y=0.95):
            ax.text(x, y, label, transform=ax.transAxes,
                ha='right', va='bottom', clip_on=False)

        add_panel_label(ax_0, '(a)')
        add_panel_label(ax_1, '(b)')
        add_panel_label(ax_2, '(c)')
        add_panel_label(ax_3, '(d)')
        add_panel_label(ax_4, '(e)')
        add_panel_label(ax_5, '(f)')

        fig.tight_layout()
        if save_dir is not None:
            save_dir = Path(save_dir)
            (save_dir / "PNG").mkdir(parents=True, exist_ok=True)
            (save_dir / "PDF").mkdir(parents=True, exist_ok=True)
            base = time.strftime("%Y-%m-%dT%H%M%S%f")
            fig.savefig(save_dir / "PNG" / f"{base}.png", dpi=200, bbox_inches="tight")
            fig.savefig(save_dir / "PDF" / f"{base}.pdf", bbox_inches="tight")

    elif fig_plot == False:
        fig = None

    def finite_or_nan(value):
        return value if np.isfinite(value) else np.nan

    fit_results = {
        'time':         time,
        'kappa_E':      finite_or_nan(kappa_E),
        'kappa_E_err':  finite_or_nan(kappa_E_err),
        'kappa_B':      finite_or_nan(kappa_B),
        'kappa_B_err':  finite_or_nan(kappa_B_err),
        'r_EB':         finite_or_nan(r_EB),
        'logrmse_EB':   finite_or_nan(logrmse_EB_abs),
        'n_EB':         n_EB if n_EB else np.nan,
        'n_EB_max':     n_EB_max if n_EB_max else np.nan,
        'f_sc_krho_low_Hz':  finite_or_nan(f_sc_krho_low),
        'f_sc_krho_high_Hz': finite_or_nan(f_sc_krho_high),

        # E64 threshold diagnostics
        'E64_frac_ge_1e-4_3_32Hz': E64_frac_over_1e4_3_32,
        'E64_n_ge_1e-4_3_32Hz':    E64_n_over_1e4_3_32,
        'E64_n_total_3_32Hz':      E64_n_total_3_32,
    }


    return fig, fit_results

In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

EB_N_JOBS = -1
EB_PARALLEL_BACKEND = "threading"
EB_FIT_RANGE = (3.0, 0.0)
EB_FOCUS_FREQ_LOW_MIN_HZ = 1e-2
PLOT_EB_EACHTIME_FIGURES = False
EB_FAST_CHUNK_SIZE = 4096
EB_FAST_MAX_CHUNK_ELEMENTS = 2_000_000

direction = eb_cfg["name"]
E2_label = eb_cfg["E2_label"]
B2_label = eb_cfg["B2_label"]
phi_label = eb_cfg["phi_label"]
EBratio_label = eb_cfg["EBratio_label"]
vsys_label = eb_cfg["vsys_label"]
Spara_label = eb_cfg["Spara_label"]

ds_64 = ds_EB64_fac_analysis_interp
ds_par = ds_parameter_interp
ds_vel = ds_velocity_ms_analysis_interp
ds_Spara = da_Spara_analysis_interp

out_dir = eb_output_dir(direction)
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "PDF").mkdir(parents=True, exist_ok=True)
(out_dir / "PNG").mkdir(parents=True, exist_ok=True)


def _as_time_freq(da):
    return np.asarray(da.transpose("time", "freq").values, dtype=float)


def _masked_log_sums(y, base_mask, x_log, min_points):
    y = np.asarray(y, dtype=float)
    valid = base_mask & np.isfinite(y) & (y > 0)
    yy = np.full(y.shape, np.nan, dtype=float)
    yy[valid] = np.log10(y[valid])

    w = valid.astype(float)
    n = w.sum(axis=1)
    sx = (w * x_log).sum(axis=1)
    sy = np.nansum(np.where(valid, yy, 0.0), axis=1)
    sxx = (w * x_log**2).sum(axis=1)
    sxy = np.nansum(np.where(valid, yy * x_log, 0.0), axis=1)
    syy = np.nansum(np.where(valid, yy**2, 0.0), axis=1)

    denom = n * sxx - sx**2
    ok = (n >= min_points) & np.isfinite(denom) & (denom > 0)

    slope = np.full(y.shape[0], np.nan, dtype=float)
    intercept = np.full(y.shape[0], np.nan, dtype=float)
    slope[ok] = (n[ok] * sxy[ok] - sx[ok] * sy[ok]) / denom[ok]
    intercept[ok] = (sy[ok] - slope[ok] * sx[ok]) / n[ok]

    sxx_centered = sxx - sx**2 / np.maximum(n, 1)
    sse = (
        syy
        + n * intercept**2
        + slope**2 * sxx
        - 2 * intercept * sy
        - 2 * slope * sxy
        + 2 * intercept * slope * sx
    )
    std_ok = ok & (n > 2) & np.isfinite(sxx_centered) & (sxx_centered > 0) & np.isfinite(sse)
    slope_std = np.full(y.shape[0], np.nan, dtype=float)
    slope_std[std_ok] = np.sqrt(np.maximum(sse[std_ok], 0.0) / (n[std_ok] - 2) / sxx_centered[std_ok])

    kappa = -slope
    return kappa, slope_std


def _corr_log_model_rows(y_obs, y_model, mask, min_points=10):
    valid = mask & np.isfinite(y_obs) & np.isfinite(y_model) & (y_obs > 0) & (y_model > 0)
    logy = np.full(y_obs.shape, np.nan, dtype=float)
    logm = np.full(y_model.shape, np.nan, dtype=float)
    logy[valid] = np.log10(y_obs[valid])
    logm[valid] = np.log10(y_model[valid])

    w = valid.astype(float)
    n = w.sum(axis=1).astype(int)
    sy = np.nansum(np.where(valid, logy, 0.0), axis=1)
    sm = np.nansum(np.where(valid, logm, 0.0), axis=1)
    syy = np.nansum(np.where(valid, logy**2, 0.0), axis=1)
    smm = np.nansum(np.where(valid, logm**2, 0.0), axis=1)
    sym = np.nansum(np.where(valid, logy * logm, 0.0), axis=1)

    cov = sym - sy * sm / np.maximum(n, 1)
    vary = syy - sy**2 / np.maximum(n, 1)
    varm = smm - sm**2 / np.maximum(n, 1)
    denom = np.sqrt(vary * varm)

    ok = (n >= min_points) & (vary > 1e-12) & (varm > 1e-12) & np.isfinite(denom) & (denom > 0)
    r = np.full(y_obs.shape[0], np.nan, dtype=float)
    r[ok] = cov[ok] / denom[ok]

    diff2 = np.nansum(np.where(valid, (logy - logm)**2, 0.0), axis=1)
    logrmse = np.full(y_obs.shape[0], np.nan, dtype=float)
    logrmse[n >= min_points] = np.sqrt(diff2[n >= min_points] / n[n >= min_points])
    return r, logrmse, n


def _compute_eb_fit_chunk(start, stop, arrays):
    freq = arrays["freq"]
    freq_row = freq[None, :]
    x_log = arrays["x_log"]
    E = arrays["E"][start:stop]
    B = arrays["B"][start:stop]
    coherency = arrays["coherency"][start:stop]
    wco_sig95 = arrays["wco_sig95"][start:stop]
    tau = arrays["tau"][start:stop]
    f_ci = arrays["f_ci"][start:stop]
    v_thi = arrays["v_thi"][start:stop]
    v_sys = arrays["v_sys"][start:stop]
    v_A = arrays["v_A"][start:stop]
    ion_mass = arrays["ion_mass"][start:stop]

    mask_coh = coherency >= wco_sig95
    with np.errstate(divide="ignore", invalid="ignore"):
        fit_high = arrays["fit_high"]
        if fit_high == 0:
            fit_high = np.sqrt(ion_mass / 9.1093837e-31 * tau)
        f_sc_krho_1 = np.abs(f_ci * v_sys / v_thi)
        f_low = f_sc_krho_1 * arrays["fit_low"]
        f_high = f_sc_krho_1 * fit_high
        mask_fit = (freq_row >= f_low[:, None]) & (freq_row <= f_high[:, None]) & (freq_row >= 1e-2)
        mask_all = mask_coh & mask_fit
        KAW_dr = (
            1.0 + (freq_row / f_ci[:, None] * v_thi[:, None] / v_sys[:, None])**2 / 2.0
        ) / np.sqrt(
            1.0
            + (freq_row / f_ci[:, None] * v_thi[:, None] / v_sys[:, None])**2 / 2.0
            * (1.0 + 1.0 / tau[:, None])
        )
        EB_ratio = np.sqrt(E / B) * 1e6 / v_A[:, None]

    kappa_E, kappa_E_err = _masked_log_sums(E, mask_all, x_log, min_points=10)
    kappa_B, kappa_B_err = _masked_log_sums(B, mask_all, x_log, min_points=10)
    r_EB, logrmse_EB, n_EB = _corr_log_model_rows(EB_ratio, KAW_dr, mask_all, min_points=10)
    _, _, n_EB_max = _corr_log_model_rows(EB_ratio, KAW_dr, mask_fit, min_points=10)

    band = arrays["band_mask"]
    E_band = E[:, band]
    finite_band = np.isfinite(E_band)
    n_total = finite_band.sum(axis=1).astype(int)
    n_over = (finite_band & (E_band >= 1e-4)).sum(axis=1).astype(int)
    frac_over = np.full(E.shape[0], np.nan, dtype=float)
    np.divide(n_over, n_total, out=frac_over, where=n_total > 0)

    return pd.DataFrame({
        "time": arrays["time"][start:stop],
        "kappa_E": kappa_E,
        "kappa_E_err": kappa_E_err,
        "kappa_B": kappa_B,
        "kappa_B_err": kappa_B_err,
        "r_EB": r_EB,
        "logrmse_EB": logrmse_EB,
        "n_EB": np.where(n_EB > 0, n_EB, np.nan),
        "n_EB_max": np.where(n_EB_max > 0, n_EB_max, np.nan),
        "f_sc_krho_low_Hz": f_low,
        "f_sc_krho_high_Hz": f_high,
        "E64_frac_ge_1e-4_3_32Hz": frac_over,
        "E64_n_ge_1e-4_3_32Hz": n_over,
        "E64_n_total_3_32Hz": n_total,
    })


def compute_eb_fit_results_fast(ds_64_window, ds_par_window, ds_vel_window, ds_Spara_window):
    ds_64_window, ds_par_window, ds_vel_window, ds_Spara_window = xr.align(
        ds_64_window,
        ds_par_window,
        ds_vel_window,
        ds_Spara_window,
        join="inner",
    )
    freq = np.asarray(ds_64_window["freq"].values, dtype=float)
    nfreq = len(freq)
    chunk_size = min(
        EB_FAST_CHUNK_SIZE,
        max(1, EB_FAST_MAX_CHUNK_ELEMENTS // max(nfreq, 1)),
    )
    arrays = {
        "time": pd.to_datetime(ds_64_window.time.values),
        "freq": freq,
        "x_log": np.log10(freq),
        "E": _as_time_freq(ds_64_window["E64"]),
        "B": _as_time_freq(ds_64_window["B64"]),
        "coherency": _as_time_freq(ds_64_window["coherency"]),
        "wco_sig95": _as_time_freq(ds_64_window["wco_sig95"]),
        "tau": np.asarray(ds_par_window["i-e_temp_ratio"].values, dtype=float),
        "ion_mass": np.asarray(ds_par_window["ion_mass_kg"].values, dtype=float),
        "f_ci": (
            np.asarray(ds_par_window["proton_cycl_freq_Hz"].values, dtype=float)
            * proton_mass_kg
            / np.asarray(ds_par_window["ion_mass_kg"].values, dtype=float)
        ),
        "v_thi": np.asarray(ds_vel_window["ion_thermal_speed"].values, dtype=float),
        "v_sys": np.asarray(ds_vel_window["perp_sys_speed"].values, dtype=float),
        "v_A": np.asarray(ds_vel_window["Alfven_speed_MID"].values, dtype=float),
        "fit_low": float(EB_FIT_RANGE[0]),
        "fit_high": float(EB_FIT_RANGE[1]),
        "band_mask": (freq >= 3.0) & (freq <= 32.0),
    }
    chunks = [(i, min(i + chunk_size, len(arrays["time"]))) for i in range(0, len(arrays["time"]), chunk_size)]
    print(
        f"Fast E/B fitting: n_time={len(arrays['time'])}, n_freq={nfreq}, "
        f"chunk_size={chunk_size}, chunks={len(chunks)}, backend={EB_PARALLEL_BACKEND}, n_jobs={EB_N_JOBS}"
    )
    with TqdmJoblib(total=len(chunks), desc=f"Fitting {direction} E/B chunks", unit="chunk"):
        frames = Parallel(n_jobs=EB_N_JOBS, backend=EB_PARALLEL_BACKEND, verbose=0)(
            delayed(_compute_eb_fit_chunk)(start, stop, arrays) for start, stop in chunks
        )
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).set_index("time").sort_index()


def process_and_save_plot(t):
    fig, fit_results = plot_freq_spectrum(
        t,
        ds_64,
        ds_par,
        ds_vel,
        ds_Spara,
        direction,
        E2_label,
        B2_label,
        phi_label,
        EBratio_label,
        vsys_label,
        Spara_label,
        fit_range=EB_FIT_RANGE,
        fig_plot=True,
    )
    if fig is None:
        return fit_results

    ts = pd.Timestamp(t)
    base = ts.strftime("%Y-%m-%dT%H%M%S%f")
    png_path = out_dir / "PNG" / f"{base}.png"
    pdf_path = out_dir / "PDF" / f"{base}.pdf"

    try:
        fig.savefig(png_path, dpi=200, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
    finally:
        plt.close(fig)

    return fit_results

t_min, t_max = pd.to_datetime(time_range)
ds_64_window = ds_64.sel(time=slice(t_min, t_max))
ds_par_window = ds_par.sel(time=slice(t_min, t_max))
ds_vel_window = ds_vel.sel(time=slice(t_min, t_max))
ds_Spara_window = ds_Spara.sel(time=slice(t_min, t_max))
time_grid = ds_64_window.time.values

if PLOT_EB_EACHTIME_FIGURES:
    with TqdmJoblib(total=len(time_grid), desc=f"Saving {direction} E/B spectra", unit="time"):
        results_list = Parallel(n_jobs=EB_N_JOBS, backend=EB_PARALLEL_BACKEND, verbose=0)(
            delayed(process_and_save_plot)(t) for t in time_grid
        )
    results_list = [r for r in results_list if r is not None]
    df_results = pd.DataFrame(results_list).set_index("time").sort_index() if results_list else pd.DataFrame()
else:
    df_results = compute_eb_fit_results_fast(ds_64_window, ds_par_window, ds_vel_window, ds_Spara_window)

if not df_results.empty:
    time_str_start = t_min.strftime("%Y%m%d_%H%M%S")
    time_str_end = t_max.strftime("%Y%m%d_%H%M%S")
    kappa_csv_filename = f"kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv"
    csv_path = out_dir / kappa_csv_filename
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")
else:
    print("No finite fitting results were produced")


In [ ]:
PLOT_EB_SINGLE_SPECTRUM = False
EB_SINGLE_SPECTRUM_TIME = pd.Timestamp(time_range[0]) + (pd.Timestamp(time_range[1]) - pd.Timestamp(time_range[0])) / 2

if PLOT_EB_SINGLE_SPECTRUM:
    fig, fit_results = plot_freq_spectrum(
        EB_SINGLE_SPECTRUM_TIME,
        ds_EB64_fac_analysis_interp,
        ds_parameter_interp,
        ds_velocity_ms_analysis_interp,
        da_Spara_analysis_interp,
        eb_cfg["name"],
        eb_cfg["E2_label"],
        eb_cfg["B2_label"],
        eb_cfg["phi_label"],
        eb_cfg["EBratio_label"],
        eb_cfg["vsys_label"],
        eb_cfg["Spara_label"],
        fit_range=EB_FIT_RANGE,
        fig_plot=True,
    )
    display(fit_results)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
import matplotlib.ticker as mticker

mpl.rcParams["font.size"] = 20

direction = eb_cfg["name"]
ds_64 = ds_EB64_fac_analysis_interp
ds_par = ds_parameter_interp
ds_vel = ds_velocity_ms_analysis_interp
S_par_64 = da_Spara_analysis_interp

S_par_label = eb_cfg["S_par_label"]
v_sys_label = eb_cfg["vsys_label"]
title_label = eb_cfg["title_label"]

out_dir = eb_output_dir(direction)

time_range_data = time_range
t_min_data, t_max_data = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime("%Y%m%d_%H%M%S")
time_str_end_data = t_max_data.strftime("%Y%m%d_%H%M%S")

kappa_csv_filename = f"kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv"
csv_path = out_dir / kappa_csv_filename

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").set_index("time").to_xarray()

EB_FOCUS_FREQ_LOW_MIN_HZ = globals().get("EB_FOCUS_FREQ_LOW_MIN_HZ", 1e-2)
if "f_sc_krho_low_Hz" not in data:
    raise KeyError(
        "kappa CSV lacks f_sc_krho_low_Hz; rerun the E/B fitting cell after this update."
    )

def true_intervals(mask, times):
    mask = np.asarray(mask, dtype=bool)
    times = pd.to_datetime(np.asarray(times))
    if len(mask) == 0:
        return []

    intervals = []
    in_block = False
    start = None

    for i, flag in enumerate(mask):
        if flag and not in_block:
            start = times[i]
            in_block = True
        elif not flag and in_block:
            intervals.append((start, times[i - 1]))
            in_block = False

    if in_block:
        intervals.append((start, times[-1]))

    return intervals


def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha="right", va="bottom", clip_on=False)


def plot_one_window(t_min, t_max):
    data_win = data.sel(time=slice(t_min, t_max))
    S_par_64_window = S_par_64.sel(time=slice(t_min, t_max))
    ds_vel_window = ds_vel.sel(time=slice(t_min, t_max))

    data_win, S_par_64_window = xr.align(
        data_win, S_par_64_window, join="inner"
    )
    _, ds_vel_window = xr.align(data_win, ds_vel_window, join="inner")

    if data_win.sizes.get("time", 0) == 0:
        print(f"Skip empty window: {t_min} - {t_max}")
        return None

    mask_data = (
        (data_win['r_EB'].data >= 0.7) &
        (data_win['logrmse_EB'].data <= 1.0) &
        (data_win['E64_frac_ge_1e-4_3_32Hz'] >= 0.8) &
        (data_win['n_EB'].data >= data_win['n_EB_max']/3.) &
        (data_win['n_EB_max'].data >= 100) &
        (data_win['f_sc_krho_low_Hz'].data > EB_FOCUS_FREQ_LOW_MIN_HZ)
    )
    good_intervals = true_intervals(mask_data.values, data_win.time.values)

    fig_kappa = plt.figure(figsize=(15, 6))
    gs = fig_kappa.add_gridspec(5, 1)
    ax_0 = fig_kappa.add_subplot(gs[0, 0])
    ax_1 = fig_kappa.add_subplot(gs[1:3, 0], sharex=ax_0)
    ax_2 = fig_kappa.add_subplot(gs[3:5, 0], sharex=ax_0)

    ax_0.tick_params(axis="x", which="both", labelbottom=False)
    ax_1.tick_params(axis="x", which="both", labelbottom=False)

    fig_kappa.suptitle(
        f"Arase, {title_label}, "
        f"{pd.Timestamp(t_min):%Y-%m-%d %H:%M:%S} - {pd.Timestamp(t_max):%H:%M:%S}"
    )

    for start, end in good_intervals:
        ax_0.axvspan(start, end, color="tab:blue", alpha=0.7, lw=0)
        ax_1.axvspan(start, end, color="tab:blue", alpha=0.2, lw=0)
        ax_2.axvspan(start, end, color="tab:blue", alpha=0.2, lw=0)

    ax_0.set_ylim(0, 1)
    ax_0.set_yticks([])
    ax_0.set_ylabel("KAW\ndetection")

    ax_1.plot(S_par_64_window.time, S_par_64_window.data * 1e3, c="k", lw=1)
    ax_1.axhline(0, color="0.5", lw=1)
    ax_1.set_ylabel(S_par_label + "\n" + r"[$\mathrm{mW/m^{2}}$]")

    ax_2.plot(
        ds_vel_window.time,
        ds_vel_window["perp_sys_speed"].data * 1e-3,
        c="k",
        lw=1,
    )
    ax_2.set_ylabel(v_sys_label + "\n" + r"[$\mathrm{km/s}$]")
    ax_2.set_xlabel("Time")

    for ax, label in zip([ax_0, ax_1, ax_2], ["(a)", "(b)", "(c)"]):
        add_panel_label(ax, label)
        ax.grid(True, alpha=0.3)

    ax_2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    fig_kappa.tight_layout()

    time_str_start = pd.Timestamp(t_min).strftime("%Y%m%d_%H%M%S")
    time_str_end = pd.Timestamp(t_max).strftime("%Y%m%d_%H%M%S")
    kappa_base = f"detection_time_{direction}_{time_str_start}_to_{time_str_end}"
    kappa_png_path = out_dir / f"{kappa_base}.png"

    fig_kappa.savefig(kappa_png_path, dpi=200, bbox_inches="tight")
    plt.close(fig_kappa)
    return kappa_png_path

start = pd.Timestamp(time_range_T[0])
end = pd.Timestamp(time_range_T[1])
step = pd.Timedelta(minutes=5)

current = start
while current < end:
    next_time = min(current + step, end)
    plot_one_window(current, next_time)
    current = next_time

plot_one_window(start, end)


## 条件を満たす時刻のランダム spectrum plot


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

EB_RANDOM_PLOT_N = 50
EB_RANDOM_PLOT_SEED = 20260717
EB_RANDOM_PLOT_DIR = eb_output_dir(eb_cfg["name"]) / "random_condition_spectra"
EB_RANDOM_PLOT_DIR.mkdir(parents=True, exist_ok=True)
(EB_RANDOM_PLOT_DIR / "PNG").mkdir(parents=True, exist_ok=True)
(EB_RANDOM_PLOT_DIR / "PDF").mkdir(parents=True, exist_ok=True)

# kappa 時系列 CSV を読み直す。直前セルを実行していなくても動くようにしておく。
direction = eb_cfg["name"]
t_min_data, t_max_data = pd.to_datetime(time_range)
time_str_start_data = t_min_data.strftime("%Y%m%d_%H%M%S")
time_str_end_data = t_max_data.strftime("%Y%m%d_%H%M%S")
kappa_csv_filename = f"kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv"
csv_path = eb_output_dir(direction) / kappa_csv_filename

data_random = pd.read_csv(csv_path)
data_random["time"] = pd.to_datetime(data_random["time"])
data_random = data_random.sort_values("time").set_index("time")
data_random = data_random.loc[t_min_data:t_max_data]

EB_FOCUS_FREQ_LOW_MIN_HZ = globals().get("EB_FOCUS_FREQ_LOW_MIN_HZ", 1e-2)
if "f_sc_krho_low_Hz" not in data_random.columns:
    raise KeyError(
        "kappa CSV lacks f_sc_krho_low_Hz; rerun the E/B fitting cell after this update."
    )

mask_random = (
    (data_random["r_EB"] >= 0.7) &
    (data_random["logrmse_EB"] <= 1.0) &
    (data_random["E64_frac_ge_1e-4_3_32Hz"] >= 0.8) &
    (data_random["n_EB"] >= data_random["n_EB_max"] / 3.0) &
    (data_random["n_EB_max"] >= 100) &
    (data_random["f_sc_krho_low_Hz"] > EB_FOCUS_FREQ_LOW_MIN_HZ)
)

candidate_times = data_random.index[mask_random].to_numpy()
n_candidates = len(candidate_times)
n_plot = min(EB_RANDOM_PLOT_N, n_candidates)

if n_plot == 0:
    print("No candidate times satisfy the random spectrum plot condition")
else:
    rng = np.random.default_rng(EB_RANDOM_PLOT_SEED)
    selected_times = np.sort(rng.choice(candidate_times, size=n_plot, replace=False))
    print(f"Random spectrum plots: selected {n_plot} / {n_candidates} candidate times")
    print(f"Output directory: {EB_RANDOM_PLOT_DIR}")

    fit_rows = []
    for t in tqdm(selected_times, desc=f"Plotting {direction} random spectra", unit="plot"):
        fig, fit_result = plot_freq_spectrum(
            t,
            ds_EB64_fac_analysis_interp,
            ds_parameter_interp,
            ds_velocity_ms_analysis_interp,
            da_Spara_analysis_interp,
            eb_cfg["name"],
            eb_cfg["E2_label"],
            eb_cfg["B2_label"],
            eb_cfg["phi_label"],
            eb_cfg["EBratio_label"],
            eb_cfg["vsys_label"],
            eb_cfg["Spara_label"],
            fit_range=EB_FIT_RANGE,
            fig_plot=True,
        )
        ts = pd.Timestamp(t)
        base = f"random_condition_spectrum_{direction}_{ts:%Y%m%d_%H%M%S_%f}"
        try:
            fig.savefig(EB_RANDOM_PLOT_DIR / "PNG" / f"{base}.png", dpi=200, bbox_inches="tight")
            fig.savefig(EB_RANDOM_PLOT_DIR / "PDF" / f"{base}.pdf", bbox_inches="tight")
        finally:
            plt.close(fig)
        fit_rows.append(fit_result)

    selected_csv_path = EB_RANDOM_PLOT_DIR / f"selected_random_times_{direction}.csv"
    pd.DataFrame(fit_rows).set_index("time").sort_index().to_csv(selected_csv_path)
    print(f"Selected fit metadata saved to {selected_csv_path}")


In [ ]:
import numpy as np
from scipy.stats import binned_statistic

def mean_by_kbin(values_2d, k_2d, k_edges):
    """
    values_2d: shape (ntime, nfreq)
    k_2d     : shape (ntime, nfreq)
    k_edges  : bin edges for kperp*rhoi

    returns:
        mean   : shape (len(k_edges)-1,)
        count  : shape (len(k_edges)-1,)
    """
    valid = (
        np.isfinite(values_2d) &
        np.isfinite(k_2d) &
        (k_2d > 0)
    )

    mean, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='mean',
        bins=k_edges,
    )

    count, _, _ = binned_statistic(
        k_2d[valid].ravel(),
        values_2d[valid].ravel(),
        statistic='count',
        bins=k_edges,
    )

    return mean, count

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq, dtype=float)
    y = np.asarray(psd, dtype=float)

    # 必ずコピーを作り、入力maskを変更しない
    msk = np.array(mask, dtype=bool, copy=True)

    msk &= (
        np.isfinite(f)
        & np.isfinite(y)
        & (f > 0)
        & (y > 0)
    )

    if np.count_nonzero(msk) < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    n = len(x)

    slope, intercept = np.polyfit(x, yy, 1)

    y_fit = intercept + slope * x
    residual = yy - y_fit

    sigma2 = np.sum(residual**2) / (n - 2)
    sxx = np.sum((x - x.mean())**2)

    if not np.isfinite(sxx) or sxx <= 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std

def _fit_kappa_from_prelogged(
    logk,
    logpsd,
    fit_mask,
    min_points=30,
):
    """
    log10(k), log10(PSD)からpower-law index kappaを高速に計算する。

    log10(PSD) = intercept - kappa * log10(k)
    """

    valid = (
        np.asarray(fit_mask, dtype=bool)
        & np.isfinite(logk)
        & np.isfinite(logpsd)
    )

    n = int(np.count_nonzero(valid))

    if n < min_points:
        return np.nan

    x = logk[valid]
    y = logpsd[valid]

    sx = np.sum(x, dtype=np.float64)
    sy = np.sum(y, dtype=np.float64)
    sxx = np.sum(x * x, dtype=np.float64)
    sxy = np.sum(x * y, dtype=np.float64)

    denominator = sxx - sx * sx / n

    if not np.isfinite(denominator) or denominator <= 0:
        return np.nan

    slope = (sxy - sx * sy / n) / denominator

    return -slope


def block_bootstrap_kappa_parallel(
    times,
    accepted_time_mask,
    frequency,
    E2_all,
    B2_all,
    f_ci_all,
    v_thi_all,
    v_sys_all,
    tau_all,
    block_duration="10s",
    n_boot=2000,
    k_lower=3.0,
    rho_e_scale_reference=None,
    recompute_rho_e_scale=True,
    min_points=30,
    seed=42,
    n_jobs=-1,
    backend="threading",
    verbose=0,
):
    """
    時間方向のnon-overlapping block bootstrapにより、
    kappa_E, kappa_Bの不確かさを並列評価する。

    Parameters
    ----------
    times : array-like of datetime64
        tmask適用前の全時間軸。shape (ntime,)

    accepted_time_mask : array-like of bool
        KAW detection criteriaを満たす時間。shape (ntime,)

    frequency : array-like
        使用周波数。shape (nfreq,)

    E2_all, B2_all : array-like
        coherence/phase criterion適用後のPSD。
        使用不可点はNaN。shape (ntime, nfreq)

    f_ci_all, v_thi_all, v_sys_all, tau_all : array-like
        各時間のプラズマパラメータ。shape (ntime,)

    block_duration : str or Timedelta
        bootstrapの時間ブロック長。

    n_boot : int
        bootstrap回数。

    k_lower : float
        fitting範囲の下限。

    rho_e_scale_reference : float or None
        recompute_rho_e_scale=Falseの場合の固定上限。

    recompute_rho_e_scale : bool
        各試行でtau平均から上限を再計算するか。

    min_points : int
        fittingに必要な最小time-frequency点数。

    seed : int
        乱数seed。

    n_jobs : int
        並列数。-1で全論理CPUを使用。

    backend : {"threading", "loky"}
        threading:
            配列を共有するため低メモリ。まずはこちらを推奨。
        loky:
            process並列。CPU並列性は高いがメモリ負荷も高い。

    verbose : int
        joblibの進捗表示。10程度で進捗を表示。

    Returns
    -------
    dict
        bootstrap分布、percentile、標準偏差など。
    """

    # --------------------------------------------------------
    # Input conversion
    # --------------------------------------------------------

    times = pd.DatetimeIndex(pd.to_datetime(times))

    accepted_time_mask = np.asarray(
        accepted_time_mask,
        dtype=bool,
    )

    frequency = np.asarray(
        frequency,
        dtype=np.float64,
    )

    E2_all = np.asarray(
        E2_all,
        dtype=np.float64,
    )

    B2_all = np.asarray(
        B2_all,
        dtype=np.float64,
    )

    f_ci_all = np.asarray(
        f_ci_all,
        dtype=np.float64,
    )

    v_thi_all = np.asarray(
        v_thi_all,
        dtype=np.float64,
    )

    v_sys_all = np.asarray(
        v_sys_all,
        dtype=np.float64,
    )

    tau_all = np.asarray(
        tau_all,
        dtype=np.float64,
    )

    ntime = len(times)
    nfreq = len(frequency)

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if ntime == 0:
        raise ValueError("times is empty")

    if accepted_time_mask.shape != (ntime,):
        raise ValueError(
            "accepted_time_mask must have shape (ntime,)"
        )

    if E2_all.shape != B2_all.shape:
        raise ValueError(
            "E2_all and B2_all must have the same shape"
        )

    if E2_all.shape != (ntime, nfreq):
        raise ValueError(
            "E2_all and B2_all must have shape (ntime, nfreq)"
        )

    for name, array in {
        "f_ci_all": f_ci_all,
        "v_thi_all": v_thi_all,
        "v_sys_all": v_sys_all,
        "tau_all": tau_all,
    }.items():
        if array.shape != (ntime,):
            raise ValueError(
                f"{name} must have shape (ntime,)"
            )

    if n_boot < 1:
        raise ValueError("n_boot must be >= 1")

    if min_points < 3:
        raise ValueError("min_points must be >= 3")

    if k_lower <= 0:
        raise ValueError("k_lower must be positive")

    block_ns = pd.to_timedelta(block_duration).value

    if block_ns <= 0:
        raise ValueError(
            "block_duration must be positive"
        )

    if not recompute_rho_e_scale:
        if (
            rho_e_scale_reference is None
            or not np.isfinite(rho_e_scale_reference)
            or rho_e_scale_reference <= 0
        ):
            raise ValueError(
                "A positive finite rho_e_scale_reference is required "
                "when recompute_rho_e_scale=False"
            )

    # --------------------------------------------------------
    # Define time blocks
    # --------------------------------------------------------

    elapsed_ns = times.asi8 - times.asi8[0]

    block_id = (
        elapsed_ns // block_ns
    ).astype(np.int64)

    n_blocks = int(block_id.max()) + 1

    # 各blockについて、最初からaccepted timeだけを保存する。
    # bootstrap内で毎回accepted_time_maskを適用する必要がなくなる。
    accepted_block_members = []

    for iblock in range(n_blocks):

        idx = np.flatnonzero(
            (block_id == iblock)
            & accepted_time_mask
        )

        accepted_block_members.append(idx)

    # --------------------------------------------------------
    # Precompute log10(k) and log10(PSD)
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):
        k_all = np.abs(
            frequency[None, :]
            / f_ci_all[:, None]
            * v_thi_all[:, None]
            / v_sys_all[:, None]
        )

    logk_all = np.full(
        k_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_k = (
        np.isfinite(k_all)
        & (k_all > 0)
    )

    logk_all[valid_k] = np.log10(
        k_all[valid_k]
    )

    # k_all自体は以後不要なので削除可能
    del k_all

    logE_all = np.full(
        E2_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_E = (
        np.isfinite(E2_all)
        & (E2_all > 0)
    )

    logE_all[valid_E] = np.log10(
        E2_all[valid_E]
    )

    logB_all = np.full(
        B2_all.shape,
        np.nan,
        dtype=np.float64,
    )

    valid_B = (
        np.isfinite(B2_all)
        & (B2_all > 0)
    )

    logB_all[valid_B] = np.log10(
        B2_all[valid_B]
    )

    logk_lower = np.log10(k_lower)

    mass_ratio = (
        1.67262192e-27
        / 9.1093837e-31
    )

    # --------------------------------------------------------
    # Generate bootstrap draws serially
    # --------------------------------------------------------

    # 並列数に依存せず、同じseedなら同じ結果を得るため、
    # block抽出乱数は並列実行前にまとめて生成する。
    rng = np.random.default_rng(seed)

    selected_blocks_all = rng.integers(
        low=0,
        high=n_blocks,
        size=(n_boot, n_blocks),
        dtype=np.int32,
    )

    # --------------------------------------------------------
    # One bootstrap realization
    # --------------------------------------------------------

    def run_one_bootstrap(selected_blocks):

        selected_parts = [
            accepted_block_members[iblock]
            for iblock in selected_blocks
            if accepted_block_members[iblock].size > 0
        ]

        if len(selected_parts) == 0:
            return np.nan, np.nan, 0

        selected_idx = np.concatenate(
            selected_parts
        )

        n_selected = selected_idx.size

        if n_selected == 0:
            return np.nan, np.nan, 0

        # --------------------------------------------
        # Upper fitting boundary
        # --------------------------------------------

        if recompute_rho_e_scale:

            tau_b = tau_all[selected_idx]

            valid_tau = (
                np.isfinite(tau_b)
                & (tau_b > 0)
            )

            if not np.any(valid_tau):
                return np.nan, np.nan, n_selected

            rho_e_scale_b = np.sqrt(
                mass_ratio
                * np.mean(tau_b[valid_tau])
            )

        else:
            rho_e_scale_b = rho_e_scale_reference

        if (
            not np.isfinite(rho_e_scale_b)
            or rho_e_scale_b <= k_lower
        ):
            return np.nan, np.nan, n_selected

        logk_upper = np.log10(
            rho_e_scale_b
        )

        # --------------------------------------------
        # Extract selected rows
        # --------------------------------------------

        logk_b = logk_all[selected_idx, :]

        fit_mask_b = (
            (logk_b >= logk_lower)
            & (logk_b <= logk_upper)
        )

        # --------------------------------------------
        # Fit E and B
        # --------------------------------------------

        kappa_E_b = _fit_kappa_from_prelogged(
            logk=logk_b,
            logpsd=logE_all[selected_idx, :],
            fit_mask=fit_mask_b,
            min_points=min_points,
        )

        kappa_B_b = _fit_kappa_from_prelogged(
            logk=logk_b,
            logpsd=logB_all[selected_idx, :],
            fit_mask=fit_mask_b,
            min_points=min_points,
        )

        return (
            kappa_E_b,
            kappa_B_b,
            n_selected,
        )

    # --------------------------------------------------------
    # Parallel execution
    # --------------------------------------------------------

    parallel_kwargs = {
        "n_jobs": n_jobs,
        "backend": backend,
        "verbose": verbose,
        "batch_size": 1,
    }

    # process並列時には大配列をmemmap経由で共有する
    if backend == "loky":
        parallel_kwargs.update({
            "max_nbytes": "50M",
            "mmap_mode": "r",
        })

    results = Parallel(
        **parallel_kwargs
    )(
        delayed(run_one_bootstrap)(
            selected_blocks_all[iboot]
        )
        for iboot in range(n_boot)
    )

    kappa_E_boot = np.asarray(
        [result[0] for result in results],
        dtype=np.float64,
    )

    kappa_B_boot = np.asarray(
        [result[1] for result in results],
        dtype=np.float64,
    )

    n_selected_time_boot = np.asarray(
        [result[2] for result in results],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    def summarize(samples):

        samples = np.asarray(
            samples,
            dtype=np.float64,
        )

        samples = samples[
            np.isfinite(samples)
        ]

        if samples.size == 0:
            return {
                "samples": samples,
                "n_valid": 0,
                "mean": np.nan,
                "median": np.nan,
                "std": np.nan,
                "q16": np.nan,
                "q84": np.nan,
                "q025": np.nan,
                "q975": np.nan,
            }

        q025, q16, q50, q84, q975 = np.percentile(
            samples,
            [2.5, 16, 50, 84, 97.5],
        )

        return {
            "samples": samples,
            "n_valid": samples.size,
            "mean": np.mean(samples),
            "median": q50,
            "std": np.std(samples, ddof=1),
            "q16": q16,
            "q84": q84,
            "q025": q025,
            "q975": q975,
        }

    return {
        "E": summarize(kappa_E_boot),
        "B": summarize(kappa_B_boot),
        "n_selected_time": n_selected_time_boot,
        "n_boot": n_boot,
        "n_blocks": n_blocks,
        "block_duration": str(block_duration),
        "n_jobs": n_jobs,
        "backend": backend,
    }

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["font.size"] = 20

# Required inputs are prepared in the E/B component setup above.
# EB_ANALYSIS_COMPONENT is currently "toroidal"; poloidal is configured but not run here.

PHASE_MODE_CONFIG = {
    "north_traveling": {
        "title": r"Northward traveling ($|\phi| \leq 60^\circ$)",
    },
    "south_traveling": {
        "title": r"Southward traveling ($|\phi| \geq 120^\circ$)",
    },
    "standing": {
        "title": r"Standing-wave-like ($60^\circ < |\phi| < 120^\circ$)",
    },
    "all": {
        "title": "All phase components",
    },
}


def make_phase_mask(phase, mode):
    """
    Select phase components using the absolute phase difference.

    north_traveling : |phi| <= 60 deg
    south_traveling : |phi| >= 120 deg
    standing        : 60 deg < |phi| < 120 deg
    all             : no phase-angle restriction, but undefined phase is excluded
    """
    if mode not in PHASE_MODE_CONFIG:
        raise ValueError(
            f"Unknown phase mode: {mode}. "
            f"Choose from {list(PHASE_MODE_CONFIG)}"
        )

    abs_phase = np.abs(np.asarray(phase, dtype=float))
    finite = np.isfinite(abs_phase)

    if mode == "north_traveling":
        return finite & (abs_phase <= 60.0)

    if mode == "south_traveling":
        return finite & (abs_phase >= 120.0)

    if mode == "standing":
        return finite & (abs_phase > 60.0) & (abs_phase < 120.0)

    return finite


def safe_log10(a):
    """Return log10(a), replacing nonpositive/nonfinite values with NaN."""
    a = np.asarray(a, dtype=float)
    out = np.full(a.shape, np.nan, dtype=float)
    valid = np.isfinite(a) & (a > 0)
    out[valid] = np.log10(a[valid])
    return out


direction = eb_cfg["name"]

E2_label = eb_cfg["E2_label"]
B2_label = eb_cfg["B2_label"]
EBratio_label = eb_cfg["EBratio_label"]
Spara_label = eb_cfg["Spara_label"]

ds_128 = ds_EB64_fac_analysis_interp
ds_par = ds_parameter_interp
ds_vel = ds_velocity_ms_analysis_interp
S_par_128 = da_Spara_analysis_interp

out_dir = eb_output_dir(direction)
out_dir.mkdir(parents=True, exist_ok=True)

time_range_data = time_range
t_min_data, t_max_data = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime("%Y%m%d_%H%M%S")
time_str_end_data = t_max_data.strftime("%Y%m%d_%H%M%S")

kappa_csv_filename = (
    f"kappa_timeseries_{direction}_"
    f"{time_str_start_data}_to_{time_str_end_data}.csv"
)
csv_path = out_dir / kappa_csv_filename

time_range_analysis = time_range
t_min, t_max = pd.to_datetime(time_range_analysis)
time_str_start = t_min.strftime("%Y%m%d_%H%M%S")
time_str_end = t_max.strftime("%Y%m%d_%H%M%S")

data = pd.read_csv(csv_path)
data["time"] = pd.to_datetime(data["time"])
if "f_sc_krho_low_Hz" not in data.columns:
    raise KeyError(
        "kappa CSV lacks f_sc_krho_low_Hz; rerun the E/B fitting cell after this update."
    )
data = data.sort_values("time").set_index("time").to_xarray()
data = data.sel(time=slice(t_min, t_max))

EB_FOCUS_FREQ_LOW_MIN_HZ = globals().get("EB_FOCUS_FREQ_LOW_MIN_HZ", 1e-2)

ds_128_window = ds_128.sel(time=slice(t_min, t_max))
ds_par_window = ds_par.sel(time=slice(t_min, t_max))
ds_vel_window = ds_vel.sel(time=slice(t_min, t_max))
S_par_128_window = S_par_128.sel(time=slice(t_min, t_max))

(
    data,
    ds_128_window,
    ds_par_window,
    ds_vel_window,
    S_par_128_window,
) = xr.align(
    data,
    ds_128_window,
    ds_par_window,
    ds_vel_window,
    S_par_128_window,
    join="inner",
)

# ----------------------------------------------------------------------
# Common masks and arrays
# ----------------------------------------------------------------------

tmask = (
    (data['r_EB'].data >= 0.7) &
    (data['logrmse_EB'].data <= 1.0) &
    (data['n_EB'].data >= data['n_EB_max']/3.) &
    (data['n_EB_max'].data >= 100) &
    (data['E64_frac_ge_1e-4_3_32Hz'] >= 0.8) &
    (data['f_sc_krho_low_Hz'].data > EB_FOCUS_FREQ_LOW_MIN_HZ)
)

fmask = ds_128_window["freq"].values >= 1e-2
frequency_mask = ds_128_window["freq"].values[fmask]

coherency_all = ds_128_window["coherency"].values[:, fmask]
wco_sig95_all = ds_128_window["wco_sig95"].values[:, fmask]
phase_all = ds_128_window["phase"].values[:, fmask]

E_all = ds_128_window["E64"].values[:, fmask]
B_all = ds_128_window["B64"].values[:, fmask]
Spara_all = ds_128_window["Spara"].values[:, fmask]

f_ci_all = ds_par_window["proton_cycl_freq_Hz"].values
v_thi_all = ds_vel_window["ion_thermal_speed"].values
v_sys_all = ds_vel_window["perp_sys_speed"].values
v_A_all = ds_vel_window["Alfven_speed_MID"].values
tau_all = ds_par_window["i-e_temp_ratio"].values
B_total_all = ds_par_window["B_total_nT"].values

f_ci_mask = f_ci_all[tmask]
v_thi_mask = v_thi_all[tmask]
v_sys_mask = v_sys_all[tmask]
v_A_mask = v_A_all[tmask]
tau_mask = tau_all[tmask]
B_total_mask = B_total_all[tmask]

with np.errstate(divide="ignore", invalid="ignore"):
    kperp_rhoi_abs = np.abs(
        frequency_mask[None, :]
        / f_ci_mask[:, None]
        * v_thi_mask[:, None]
        / v_sys_mask[:, None]
    )

valid_k = np.isfinite(kperp_rhoi_abs) & (kperp_rhoi_abs > 0)
if not np.any(valid_k):
    raise RuntimeError("No finite positive |kx| rho_i values were obtained.")

kmin = np.nanmin(kperp_rhoi_abs[valid_k])
kmax = np.nanmax(kperp_rhoi_abs[valid_k])

nkbin = 100
k_edges = np.logspace(np.log10(kmin), np.log10(kmax), nkbin + 1)
k_centers = np.sqrt(k_edges[:-1] * k_edges[1:])

valid_tau = np.isfinite(tau_mask) & (tau_mask > 0)
if not np.any(valid_tau):
    raise RuntimeError("No finite positive ion-to-electron temperature ratios.")

mass_ratio = 1.67262192e-27 / 9.1093837e-31
rho_e_scale = np.sqrt(mass_ratio * np.mean(tau_mask[valid_tau]))

kappa_mask = (
    (kperp_rhoi_abs >= 3.0)
    & (kperp_rhoi_abs <= rho_e_scale)
)


def run_one_phase_mode(
    phase_mode,
    *,
    n_boot=2000,
    block_duration="10s",
    seed=42,
    save_pdf=False,
):
    """Calculate, bootstrap, and plot one phase-component selection."""
    phase_title = PHASE_MODE_CONFIG[phase_mode]["title"]

    # Apply exactly the same phase/coherence rule to the ordinary fit
    # and to the bootstrap input.
    phase_condition_all = make_phase_mask(phase_all, phase_mode)
    cmask_all = (
        (coherency_all >= wco_sig95_all)
        & phase_condition_all
    )

    E_all_for_fit = np.where(cmask_all, E_all, np.nan)
    B_all_for_fit = np.where(cmask_all, B_all, np.nan)

    cmask = cmask_all[tmask]
    E_mask = np.where(cmask, E_all[tmask], np.nan)
    B_mask = np.where(cmask, B_all[tmask], np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        Spara_freq_mask = (
            Spara_all[tmask]
            * 1e3
            / B_total_mask[:, None]
        )
        Spara_freq_mask = np.abs(
            np.where(cmask, Spara_freq_mask, np.nan)
        )

        EB_ratio_mask = (
            np.sqrt(E_mask / B_mask)
            / v_A_mask[:, None]
            * 1e6
        )

        W_wave_mask = (
            0.5
            / mu_0
            / v_A_mask[:, None] ** 2
            / (1.0 + 0.5 * kperp_rhoi_abs ** 2)
            * E_mask
            * 1e-6
            / elementary_charge
            * 1e-6
        )

    log10_E_mean_k, E_count_k = mean_by_kbin(
        safe_log10(E_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_B_mean_k, B_count_k = mean_by_kbin(
        safe_log10(B_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_EB_ratio_mean_k, EB_ratio_count_k = mean_by_kbin(
        safe_log10(EB_ratio_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_Spara_mean_k, Spara_count_k = mean_by_kbin(
        safe_log10(Spara_freq_mask),
        kperp_rhoi_abs,
        k_edges,
    )
    log10_W_wave_mean_k, W_wave_count_k = mean_by_kbin(
        safe_log10(W_wave_mask),
        kperp_rhoi_abs,
        k_edges,
    )

    kappa_E, kappa_E_err_formal = _fit_powerlaw_kappa(
        kperp_rhoi_abs,
        E_mask,
        kappa_mask,
        min_points=30,
    )
    kappa_B, kappa_B_err_formal = _fit_powerlaw_kappa(
        kperp_rhoi_abs,
        B_mask,
        kappa_mask,
        min_points=30,
    )

    bootstrap_result = block_bootstrap_kappa_parallel(
        times=data["time"].values,
        accepted_time_mask=np.asarray(tmask, dtype=bool),
        frequency=frequency_mask,
        E2_all=E_all_for_fit,
        B2_all=B_all_for_fit,
        f_ci_all=f_ci_all,
        v_thi_all=v_thi_all,
        v_sys_all=v_sys_all,
        tau_all=tau_all,
        block_duration=block_duration,
        n_boot=n_boot,
        k_lower=3.0,
        recompute_rho_e_scale=True,
        rho_e_scale_reference=rho_e_scale,
        min_points=30,
        seed=seed,

        # 全論理CPUを使用
        n_jobs=-1,
    
        # 大配列を共有する
        backend="threading",
    
        # 進捗表示
        verbose=10,
    )

    E_boot = bootstrap_result["E"]
    B_boot = bootstrap_result["B"]

    print(f"\n[{phase_mode}] {phase_title}")
    print(
        f"E: kappa={kappa_E:.4f}, formal={kappa_E_err_formal:.4g}, "
        f"bootstrap 16--84%=[{E_boot['q16']:.4f}, {E_boot['q84']:.4f}]"
    )
    print(
        f"B: kappa={kappa_B:.4f}, formal={kappa_B_err_formal:.4g}, "
        f"bootstrap 16--84%=[{B_boot['q16']:.4f}, {B_boot['q84']:.4f}]"
    )

    # ------------------------------------------------------------------
    # Main figure
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(10, 16))
    gs = fig.add_gridspec(4, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis="x", which="both", labelbottom=False)
    ax_1.tick_params(axis="x", which="both", labelbottom=False)
    ax_2.tick_params(axis="x", which="both", labelbottom=False)

    ax_0.scatter(
        kperp_rhoi_abs,
        E_mask,
        s=0.05,
        alpha=1 / 51,
        c="lime",
    )
    ax_0.plot(
        k_centers,
        10 ** log10_E_mean_k,
        lw=2,
        linestyle="solid",
        c="green",
    )
    ax_0.scatter(
        kperp_rhoi_abs,
        B_mask,
        s=0.05,
        alpha=1 / 51,
        c="violet",
    )
    ax_0.plot(
        k_centers,
        10 ** log10_B_mean_k,
        lw=2,
        linestyle="solid",
        c="purple",
    )
    ax_0.set_yscale("log")
    ax_0.set_ylim(1e-6, 1e4)
    ax_0.set_yticks(
        [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
         1e0, 1e1, 1e2, 1e3, 1e4]
    )
    ax_0.set_xscale("log")
    ax_0.set_xlim(1e-1, 4e2)
    ax_0.minorticks_on()
    ax_0.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_0.set_ylabel(
        E2_label
        + r" [$\mathrm{(mV/m)^{2}/Hz}$]"
        + "\n"
        + B2_label
        + r" [$\mathrm{nT^{2}/Hz}$]"
    )

    if np.isfinite(kappa_E):
        E_err_minus = kappa_E - E_boot["q16"]
        E_err_plus = E_boot["q84"] - kappa_E
        txt_E = (
            rf"$\kappa_E={kappa_E:.2f}"
            rf"^{{+{E_err_plus:.2f}}}"
            rf"_{{-{E_err_minus:.2f}}}$"
        )
        ax_0.text(
            0.02,
            0.15,
            txt_E,
            transform=ax_0.transAxes,
            ha="left",
            va="bottom",
            color="green",
            bbox=dict(
                facecolor="white",
                alpha=0.7,
                edgecolor="none",
                pad=2,
            ),
        )

    if np.isfinite(kappa_B):
        B_err_minus = kappa_B - B_boot["q16"]
        B_err_plus = B_boot["q84"] - kappa_B
        txt_B = (
            rf"$\kappa_B={kappa_B:.2f}"
            rf"^{{+{B_err_plus:.2f}}}"
            rf"_{{-{B_err_minus:.2f}}}$"
        )
        ax_0.text(
            0.02,
            0.02,
            txt_B,
            transform=ax_0.transAxes,
            ha="left",
            va="bottom",
            color="purple",
            bbox=dict(
                facecolor="white",
                alpha=0.7,
                edgecolor="none",
                pad=2,
            ),
        )

    ax_1.scatter(
        kperp_rhoi_abs,
        EB_ratio_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_1.plot(
        k_centers,
        10 ** log10_EB_ratio_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_1.set_yscale("log")
    ax_1.set_ylim(1e-1, 1e2)
    ax_1.set_yticks([1e-1, 1e0, 1e1, 1e2])
    ax_1.minorticks_on()
    ax_1.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_1.set_ylabel(EBratio_label)

    ax_2.scatter(
        kperp_rhoi_abs,
        Spara_freq_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_2.plot(
        k_centers,
        10 ** log10_Spara_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_2.set_yscale("log")
    ax_2.set_ylim(1e-9, 1e-3)
    ax_2.set_yticks([1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3])
    ax_2.minorticks_on()
    ax_2.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_2.set_ylabel(
        Spara_label
        + r"$/B_{0}$"
        + "\n"
        + r"[$\mathrm{mW}/\mathrm{m}^{2}/\mathrm{nT}$]"
    )

    ax_3.scatter(
        kperp_rhoi_abs,
        W_wave_mask,
        s=0.05,
        alpha=1 / 51,
        c="gray",
    )
    ax_3.plot(
        k_centers,
        10 ** log10_W_wave_mean_k,
        lw=2,
        linestyle="solid",
        c="k",
    )
    ax_3.set_yscale("log")
    ax_3.set_ylim(1e-7, 1e3)
    ax_3.set_yticks(
        [1e-7, 1e-6, 1e-5, 1e-4, 1e-3,
         1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3]
    )
    ax_3.minorticks_on()
    ax_3.grid(which="both", alpha=0.3, linestyle="dashed")
    ax_3.set_ylabel(r"$W$ [$\mathrm{eV}/\mathrm{cm}^{3}/\mathrm{Hz}$]")
    ax_3.set_xlabel(r"$\left| k_{x} \right| \rho_{\mathrm{i}}$")

    ax_0.set_title(
        f"Arase, {phase_title}\n"
        f"{t_min.strftime('%H:%M:%S.%f')} - "
        f"{t_max.strftime('%H:%M:%S.%f')}"
    )

    def add_panel_label(ax, label, x=-0.10, y=0.95):
        ax.text(
            x,
            y,
            label,
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            clip_on=False,
        )

    for ax in [ax_0, ax_1, ax_2, ax_3]:
        ax.axvline(
            3,
            lw=2,
            linestyle="dashed",
            c="magenta",
            alpha=0.7,
        )
        ax.axvline(
            rho_e_scale,
            lw=2,
            linestyle="dashed",
            c="magenta",
            alpha=0.7,
        )

    fig.tight_layout()

    kappa_base = (
        f"kperp_rhoi_averaged_{direction}_{phase_mode}_"
        f"{time_str_start}_to_{time_str_end}"
    )
    main_png_path = out_dir / (kappa_base + ".png")
    main_pdf_path = out_dir / (kappa_base + ".pdf")

    fig.savefig(main_png_path, dpi=200, bbox_inches="tight")
    if save_pdf:
        fig.savefig(main_pdf_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {main_png_path}")

    # ------------------------------------------------------------------
    # Bootstrap histogram
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(
        E_boot["samples"],
        bins=40,
        alpha=0.5,
        label=r"$\kappa_E$",
    )
    ax.hist(
        B_boot["samples"],
        bins=40,
        alpha=0.5,
        label=r"$\kappa_B$",
    )
    ax.axvline(kappa_E, linestyle="--")
    ax.axvline(kappa_B, linestyle="--")
    ax.set_xlabel(r"$\kappa$")
    ax.set_ylabel("Count")
    ax.set_title(phase_title)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()

    bootstrap_base = kappa_base + "_bootstrap"
    bootstrap_png_path = out_dir / (bootstrap_base + ".png")
    fig.savefig(
        bootstrap_png_path,
        dpi=200,
        bbox_inches="tight",
    )
    plt.close(fig)
    print(f"Saved to {bootstrap_png_path}")

    n_phase_points = int(np.count_nonzero(cmask))
    n_coherent_points = int(
        np.count_nonzero((coherency_all[tmask] >= wco_sig95_all[tmask]))
    )

    return {
        "phase_mode": phase_mode,
        "phase_label": phase_title,
        "n_phase_coherent_points": n_phase_points,
        "n_all_coherent_points": n_coherent_points,
        "phase_fraction_of_coherent": (
            n_phase_points / n_coherent_points
            if n_coherent_points > 0
            else np.nan
        ),
        "kappa_E": kappa_E,
        "kappa_E_formal_err": kappa_E_err_formal,
        "kappa_E_boot_median": E_boot["median"],
        "kappa_E_boot_std": E_boot["std"],
        "kappa_E_q16": E_boot["q16"],
        "kappa_E_q84": E_boot["q84"],
        "kappa_B": kappa_B,
        "kappa_B_formal_err": kappa_B_err_formal,
        "kappa_B_boot_median": B_boot["median"],
        "kappa_B_boot_std": B_boot["std"],
        "kappa_B_q16": B_boot["q16"],
        "kappa_B_q84": B_boot["q84"],
        "main_figure": main_png_path,
        "bootstrap_figure": bootstrap_png_path,
    }


# ----------------------------------------------------------------------
# Run all four phase selections
# ----------------------------------------------------------------------

phase_modes = [
    "north_traveling",
    "south_traveling",
    "standing",
    "all",
]

results = []

for phase_mode in phase_modes:
    result = run_one_phase_mode(
        phase_mode,
        n_boot=2000,
        block_duration="10s",
        seed=42,
        save_pdf=False,
    )
    results.append(result)

summary_df = pd.DataFrame(results)

summary_filename = (
    f"kappa_phase_summary_{direction}_"
    f"{time_str_start}_to_{time_str_end}.csv"
)
summary_path = out_dir / summary_filename
summary_df.to_csv(summary_path, index=False)

print("\nSummary")
print(
    summary_df[
        [
            "phase_mode",
            "n_phase_coherent_points",
            "phase_fraction_of_coherent",
            "kappa_E",
            "kappa_E_q16",
            "kappa_E_q84",
            "kappa_B",
            "kappa_B_q16",
            "kappa_B_q84",
        ]
    ].to_string(index=False)
)
print(f"\nSaved summary to {summary_path}")